# Complete IBM AML transaction-level experiment

Run this notebook from top to bottom in a Kaggle GPU session after attaching the IBM AML dataset. The workflow saves checkpoints under `/kaggle/working/aml_outputs` so interrupted sessions can resume.

The values stored under the historical field name `auprc` are calculated with scikit-learn `average_precision_score` and should be interpreted as Average Precision.


## Dataset discovery and preview

Attach the IBM AML dataset before running this cell.


In [ ]:
from pathlib import Path
import pandas as pd

# Find only the HI-Small transaction file
matches = list(
    Path("/kaggle/input").rglob("HI-Small_Trans.csv")
)

if not matches:
    raise FileNotFoundError(
        "HI-Small_Trans.csv was not found. Check that the IBM AML dataset is attached."
    )

DATA_PATH = matches[0]

print("Using file:", DATA_PATH)
print(
    "File size in GB:",
    round(DATA_PATH.stat().st_size / (1024 ** 3), 2)
)

# Read only five rows for now
preview = pd.read_csv(DATA_PATH, nrows=5)

print("\nColumns:")
print(preview.columns.tolist())

print("\nFirst five rows:")
display(preview)

## Complete dataset audit

Loads the full transaction file and records the raw audit.


In [ ]:
from pathlib import Path
import json
import pandas as pd

OUTPUT_DIR = Path("/kaggle/working/aml_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DTYPES = {
    "From Bank": "int32",
    "Account": "string",
    "To Bank": "int32",
    "Account.1": "string",
    "Amount Received": "float64",
    "Receiving Currency": "category",
    "Amount Paid": "float64",
    "Payment Currency": "category",
    "Payment Format": "category",
    "Is Laundering": "int8",
}

print("Loading the complete dataset...")
df = pd.read_csv(
    DATA_PATH,
    dtype=DTYPES,
    low_memory=False,
)

print("Parsing timestamps...")
df["Timestamp"] = pd.to_datetime(
    df["Timestamp"],
    format="%Y/%m/%d %H:%M",
    errors="coerce",
)

row_count = int(len(df))
column_count = int(len(df.columns))
duplicate_count = int(df.duplicated().sum())
missing_by_column = df.isna().sum()
label_counts = df["Is Laundering"].value_counts().sort_index()

legitimate_count = int(label_counts.get(0, 0))
laundering_count = int(label_counts.get(1, 0))
laundering_rate = laundering_count / row_count

timestamp_min = df["Timestamp"].min()
timestamp_max = df["Timestamp"].max()

column_summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_values": missing_by_column,
    "unique_values": df.nunique(dropna=True),
})

daily_counts = (
    df.groupby(df["Timestamp"].dt.date)["Is Laundering"]
    .agg(total_transactions="size", laundering_transactions="sum")
    .reset_index()
    .rename(columns={"Timestamp": "date"})
)

daily_counts["legitimate_transactions"] = (
    daily_counts["total_transactions"]
    - daily_counts["laundering_transactions"]
)

daily_counts["laundering_percent"] = (
    100
    * daily_counts["laundering_transactions"]
    / daily_counts["total_transactions"]
)

audit = {
    "source_file": str(DATA_PATH),
    "rows": row_count,
    "columns": column_count,
    "duplicate_rows": duplicate_count,
    "legitimate_transactions": legitimate_count,
    "laundering_transactions": laundering_count,
    "laundering_rate": laundering_rate,
    "timestamp_min": str(timestamp_min),
    "timestamp_max": str(timestamp_max),
}

column_summary.to_csv(
    OUTPUT_DIR / "column_summary.csv",
    index=True,
)

daily_counts.to_csv(
    OUTPUT_DIR / "daily_class_counts.csv",
    index=False,
)

with open(
    OUTPUT_DIR / "dataset_audit.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(audit, file, indent=2)

print("\nDATASET AUDIT")
print("Rows:", f"{row_count:,}")
print("Columns:", column_count)
print("Duplicate rows:", f"{duplicate_count:,}")
print("Legitimate transactions:", f"{legitimate_count:,}")
print("Laundering transactions:", f"{laundering_count:,}")
print("Laundering percentage:", f"{100 * laundering_rate:.6f}%")
print("Earliest timestamp:", timestamp_min)
print("Latest timestamp:", timestamp_max)

print("\nMISSING VALUES")
print(missing_by_column)

print("\nCOLUMN SUMMARY")
display(column_summary)

print("\nTRANSACTIONS BY DATE")
display(daily_counts)

print("\nSaved audit files in:", OUTPUT_DIR)

## Duplicate removal and chronological split

Keeps identical timestamps in a single partition.


In [ ]:
import numpy as np
import pandas as pd

print("Removing exact duplicate rows...")

rows_before = len(df)
df = df.drop_duplicates(ignore_index=True)
rows_after = len(df)

print("Rows before:", f"{rows_before:,}")
print("Rows after:", f"{rows_after:,}")
print("Rows removed:", f"{rows_before - rows_after:,}")

print("\nSorting transactions chronologically...")

df = (
    df.sort_values(
        "Timestamp",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

# Give every retained transaction a stable identifier
df.insert(
    0,
    "Transaction ID",
    np.arange(len(df), dtype=np.int64),
)

# Count transactions at every unique timestamp
timestamp_counts = (
    df.groupby("Timestamp", sort=True)
    .size()
)

cumulative_counts = timestamp_counts.cumsum()

train_target = 0.60 * len(df)
validation_end_target = 0.80 * len(df)

# Select boundaries closest to 60 percent and 80 percent
train_end = (
    cumulative_counts
    .sub(train_target)
    .abs()
    .idxmin()
)

validation_end = (
    cumulative_counts
    .sub(validation_end_target)
    .abs()
    .idxmin()
)

split_codes = np.full(
    len(df),
    2,
    dtype=np.int8,
)

split_codes[
    df["Timestamp"].le(train_end).to_numpy()
] = 0

split_codes[
    (
        df["Timestamp"].gt(train_end)
        & df["Timestamp"].le(validation_end)
    ).to_numpy()
] = 1

df["Split"] = pd.Categorical.from_codes(
    split_codes,
    categories=["train", "validation", "test"],
    ordered=True,
)

# Verify that one timestamp never appears in two partitions
maximum_partitions_per_timestamp = (
    df.groupby("Timestamp", observed=True)["Split"]
    .nunique()
    .max()
)

assert maximum_partitions_per_timestamp == 1

split_summary = (
    df.groupby("Split", observed=True)
    .agg(
        transactions=("Is Laundering", "size"),
        laundering_transactions=("Is Laundering", "sum"),
        first_timestamp=("Timestamp", "min"),
        last_timestamp=("Timestamp", "max"),
    )
    .reset_index()
)

split_summary["legitimate_transactions"] = (
    split_summary["transactions"]
    - split_summary["laundering_transactions"]
)

split_summary["laundering_percent"] = (
    100
    * split_summary["laundering_transactions"]
    / split_summary["transactions"]
)

split_summary["dataset_percent"] = (
    100
    * split_summary["transactions"]
    / len(df)
)

tail_start = pd.Timestamp("2022-09-11 00:00:00")

tail_summary = (
    df.assign(
        period=np.where(
            df["Timestamp"] < tail_start,
            "dense_period",
            "timestamp_tail",
        )
    )
    .groupby("period")
    .agg(
        transactions=("Is Laundering", "size"),
        laundering_transactions=("Is Laundering", "sum"),
        first_timestamp=("Timestamp", "min"),
        last_timestamp=("Timestamp", "max"),
    )
    .reset_index()
)

tail_summary["laundering_percent"] = (
    100
    * tail_summary["laundering_transactions"]
    / tail_summary["transactions"]
)

split_summary.to_csv(
    OUTPUT_DIR / "split_summary.csv",
    index=False,
)

tail_summary.to_csv(
    OUTPUT_DIR / "timestamp_tail_summary.csv",
    index=False,
)

df[
    [
        "Transaction ID",
        "Timestamp",
        "Split",
        "Is Laundering",
    ]
].to_parquet(
    OUTPUT_DIR / "split_manifest.parquet",
    index=False,
)

print("\nTrain ends at:", train_end)
print("Validation ends at:", validation_end)

print("\nSPLIT SUMMARY")
display(split_summary)

print("\nTIMESTAMP TAIL SUMMARY")
display(tail_summary)

print(
    "\nMaximum partitions used by one timestamp:",
    maximum_partitions_per_timestamp,
)

print("\nSaved split files in:", OUTPUT_DIR)

## Account node mapping

Creates composite bank-account identifiers for graph connectivity.


In [ ]:
import gc
import numpy as np
import pandas as pd

print("Creating composite account identifiers...")

row_count = len(df)

source_accounts = pd.MultiIndex.from_arrays(
    [
        df["From Bank"].to_numpy(),
        df["Account"].to_numpy(),
    ],
    names=["Bank", "Account ID"],
)

destination_accounts = pd.MultiIndex.from_arrays(
    [
        df["To Bank"].to_numpy(),
        df["Account.1"].to_numpy(),
    ],
    names=["Bank", "Account ID"],
)

all_accounts = source_accounts.append(
    destination_accounts
)

node_codes, unique_accounts = pd.factorize(
    all_accounts,
    sort=True,
)

source_node_ids = node_codes[:row_count]
destination_node_ids = node_codes[row_count:]

if node_codes.min() < 0:
    raise ValueError(
        "A missing account identifier was found."
    )

if len(unique_accounts) >= np.iinfo(np.int32).max:
    raise ValueError(
        "The graph contains too many nodes for int32 identifiers."
    )

df["Source Node ID"] = source_node_ids.astype(
    np.int32
)

df["Destination Node ID"] = (
    destination_node_ids.astype(np.int32)
)

node_mapping = unique_accounts.to_frame(
    index=False
)

node_mapping.insert(
    0,
    "Node ID",
    np.arange(
        len(node_mapping),
        dtype=np.int32,
    ),
)

self_transfer_count = int(
    (
        df["Source Node ID"]
        == df["Destination Node ID"]
    ).sum()
)

same_bank_count = int(
    (
        df["From Bank"]
        == df["To Bank"]
    ).sum()
)

node_mapping.to_parquet(
    OUTPUT_DIR / "account_node_mapping.parquet",
    index=False,
)

df[
    [
        "Transaction ID",
        "Timestamp",
        "Source Node ID",
        "Destination Node ID",
        "Split",
        "Is Laundering",
    ]
].to_parquet(
    OUTPUT_DIR / "edge_identity_manifest.parquet",
    index=False,
)

del source_accounts
del destination_accounts
del all_accounts
del node_codes
del source_node_ids
del destination_node_ids
del unique_accounts

gc.collect()

print("\nACCOUNT NODE SUMMARY")
print("Unique account nodes:", f"{len(node_mapping):,}")
print("Self-transfer transactions:", f"{self_transfer_count:,}")
print("Same-bank transactions:", f"{same_bank_count:,}")

print("\nNode mapping preview:")
display(node_mapping.head())

print("\nSaved:")
print(
    OUTPUT_DIR / "account_node_mapping.parquet"
)
print(
    OUTPUT_DIR / "edge_identity_manifest.parquet"
)

## Past-only feature engineering

Builds transaction and strictly historical aggregate features.


In [ ]:
import gc
import json
import numpy as np
import pandas as pd

print("Checking transaction amounts...")

if (df["Amount Paid"] < 0).any():
    raise ValueError("Negative Amount Paid values were found.")

if (df["Amount Received"] < 0).any():
    raise ValueError("Negative Amount Received values were found.")

print("Creating basic transaction features...")

df["log_amount_paid"] = np.log1p(
    df["Amount Paid"]
).astype(np.float32)

df["log_amount_received"] = np.log1p(
    df["Amount Received"]
).astype(np.float32)

df["same_bank"] = (
    df["From Bank"] == df["To Bank"]
).astype(np.int8)

df["self_transfer"] = (
    df["Source Node ID"]
    == df["Destination Node ID"]
).astype(np.int8)

df["same_currency"] = (
    df["Payment Currency"]
    == df["Receiving Currency"]
).astype(np.int8)

minute_of_day = (
    df["Timestamp"].dt.hour * 60
    + df["Timestamp"].dt.minute
)

weekday = df["Timestamp"].dt.dayofweek

df["time_sin"] = np.sin(
    2 * np.pi * minute_of_day / 1440
).astype(np.float32)

df["time_cos"] = np.cos(
    2 * np.pi * minute_of_day / 1440
).astype(np.float32)

df["weekday_sin"] = np.sin(
    2 * np.pi * weekday / 7
).astype(np.float32)

df["weekday_cos"] = np.cos(
    2 * np.pi * weekday / 7
).astype(np.float32)

del minute_of_day
del weekday
gc.collect()

print("Creating sender history counts...")

sender_all_rank = (
    df.groupby(
        "Source Node ID",
        sort=False,
    )
    .cumcount()
)

sender_same_time_rank = (
    df.groupby(
        [
            "Source Node ID",
            "Timestamp",
        ],
        sort=False,
    )
    .cumcount()
)

df["sender_prior_tx_count"] = (
    sender_all_rank - sender_same_time_rank
).astype(np.int32)

del sender_all_rank
del sender_same_time_rank
gc.collect()

print("Creating receiver history counts...")

receiver_all_rank = (
    df.groupby(
        "Destination Node ID",
        sort=False,
    )
    .cumcount()
)

receiver_same_time_rank = (
    df.groupby(
        [
            "Destination Node ID",
            "Timestamp",
        ],
        sort=False,
    )
    .cumcount()
)

df["receiver_prior_tx_count"] = (
    receiver_all_rank - receiver_same_time_rank
).astype(np.int32)

del receiver_all_rank
del receiver_same_time_rank
gc.collect()

print("Creating sender currency-specific history...")

sender_currency_groups = [
    "Source Node ID",
    "Payment Currency",
]

sender_currency_time_groups = [
    "Source Node ID",
    "Payment Currency",
    "Timestamp",
]

sender_currency_rank = (
    df.groupby(
        sender_currency_groups,
        sort=False,
        observed=True,
    )
    .cumcount()
)

sender_currency_time_rank = (
    df.groupby(
        sender_currency_time_groups,
        sort=False,
        observed=True,
    )
    .cumcount()
)

df["sender_prior_currency_count"] = (
    sender_currency_rank
    - sender_currency_time_rank
).astype(np.int32)

sender_amount_cumulative = (
    df.groupby(
        sender_currency_groups,
        sort=False,
        observed=True,
    )["Amount Paid"]
    .cumsum()
)

sender_amount_same_time = (
    df.groupby(
        sender_currency_time_groups,
        sort=False,
        observed=True,
    )["Amount Paid"]
    .cumsum()
)

df["sender_prior_currency_amount"] = (
    sender_amount_cumulative
    - sender_amount_same_time
).clip(lower=0)

sender_count_array = (
    df["sender_prior_currency_count"]
    .to_numpy(dtype=np.float64)
)

sender_amount_array = (
    df["sender_prior_currency_amount"]
    .to_numpy(dtype=np.float64)
)

sender_mean = np.zeros(
    len(df),
    dtype=np.float64,
)

np.divide(
    sender_amount_array,
    sender_count_array,
    out=sender_mean,
    where=sender_count_array > 0,
)

df["sender_prior_currency_mean"] = sender_mean

del sender_currency_rank
del sender_currency_time_rank
del sender_amount_cumulative
del sender_amount_same_time
del sender_count_array
del sender_amount_array
del sender_mean
gc.collect()

print("Creating receiver currency-specific history...")

receiver_currency_groups = [
    "Destination Node ID",
    "Receiving Currency",
]

receiver_currency_time_groups = [
    "Destination Node ID",
    "Receiving Currency",
    "Timestamp",
]

receiver_currency_rank = (
    df.groupby(
        receiver_currency_groups,
        sort=False,
        observed=True,
    )
    .cumcount()
)

receiver_currency_time_rank = (
    df.groupby(
        receiver_currency_time_groups,
        sort=False,
        observed=True,
    )
    .cumcount()
)

df["receiver_prior_currency_count"] = (
    receiver_currency_rank
    - receiver_currency_time_rank
).astype(np.int32)

receiver_amount_cumulative = (
    df.groupby(
        receiver_currency_groups,
        sort=False,
        observed=True,
    )["Amount Received"]
    .cumsum()
)

receiver_amount_same_time = (
    df.groupby(
        receiver_currency_time_groups,
        sort=False,
        observed=True,
    )["Amount Received"]
    .cumsum()
)

df["receiver_prior_currency_amount"] = (
    receiver_amount_cumulative
    - receiver_amount_same_time
).clip(lower=0)

receiver_count_array = (
    df["receiver_prior_currency_count"]
    .to_numpy(dtype=np.float64)
)

receiver_amount_array = (
    df["receiver_prior_currency_amount"]
    .to_numpy(dtype=np.float64)
)

receiver_mean = np.zeros(
    len(df),
    dtype=np.float64,
)

np.divide(
    receiver_amount_array,
    receiver_count_array,
    out=receiver_mean,
    where=receiver_count_array > 0,
)

df["receiver_prior_currency_mean"] = receiver_mean

del receiver_currency_rank
del receiver_currency_time_rank
del receiver_amount_cumulative
del receiver_amount_same_time
del receiver_count_array
del receiver_amount_array
del receiver_mean
gc.collect()

print("Applying logarithmic transformations...")

history_columns = [
    "sender_prior_tx_count",
    "receiver_prior_tx_count",
    "sender_prior_currency_count",
    "sender_prior_currency_amount",
    "sender_prior_currency_mean",
    "receiver_prior_currency_count",
    "receiver_prior_currency_amount",
    "receiver_prior_currency_mean",
]

for column in history_columns:
    df["log_" + column] = np.log1p(
        df[column]
    ).astype(np.float32)

numeric_features = [
    "log_amount_paid",
    "log_amount_received",
    "same_bank",
    "self_transfer",
    "same_currency",
    "time_sin",
    "time_cos",
    "weekday_sin",
    "weekday_cos",
    "log_sender_prior_tx_count",
    "log_receiver_prior_tx_count",
    "log_sender_prior_currency_count",
    "log_sender_prior_currency_amount",
    "log_sender_prior_currency_mean",
    "log_receiver_prior_currency_count",
    "log_receiver_prior_currency_amount",
    "log_receiver_prior_currency_mean",
]

categorical_features = [
    "Payment Currency",
    "Receiving Currency",
    "Payment Format",
]

tabular_features = (
    numeric_features
    + categorical_features
)

forbidden_predictors = {
    "Transaction ID",
    "Timestamp",
    "From Bank",
    "Account",
    "To Bank",
    "Account.1",
    "Source Node ID",
    "Destination Node ID",
    "Is Laundering",
    "Split",
}

unexpected_features = (
    set(tabular_features)
    & forbidden_predictors
)

if unexpected_features:
    raise ValueError(
        "Forbidden predictors found: "
        + str(sorted(unexpected_features))
    )

if df[tabular_features].isna().any().any():
    raise ValueError(
        "Missing values were created in the features."
    )

feature_config = {
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "excluded_raw_identifiers": [
        "From Bank",
        "Account",
        "To Bank",
        "Account.1",
    ],
    "target": "Is Laundering",
    "split_column": "Split",
    "history_rule": (
        "Only transactions with an earlier timestamp "
        "contribute to historical features."
    ),
}

with open(
    OUTPUT_DIR / "feature_configuration.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        feature_config,
        file,
        indent=2,
    )

checkpoint_columns = [
    "Transaction ID",
    "Timestamp",
    "Source Node ID",
    "Destination Node ID",
    "Split",
    "Is Laundering",
] + tabular_features

feature_checkpoint = (
    OUTPUT_DIR / "modeling_features.parquet"
)

df[checkpoint_columns].to_parquet(
    feature_checkpoint,
    index=False,
)

print("\nFEATURE SUMMARY")
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Total model features:", len(tabular_features))
print(
    "Checkpoint size in GB:",
    round(
        feature_checkpoint.stat().st_size
        / (1024 ** 3),
        3,
    ),
)

print("\nModel features:")
for feature in tabular_features:
    print(" ", feature)

print("\nHistorical feature preview:")
display(
    df[
        [
            "Timestamp",
            "sender_prior_tx_count",
            "receiver_prior_tx_count",
            "sender_prior_currency_count",
            "receiver_prior_currency_count",
        ]
    ].sample(
        10,
        random_state=42,
    )
)

## Training samples

Creates the common seed-specific training transaction samples.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

SEEDS = [42, 52, 62, 72, 82]
NEGATIVES_PER_POSITIVE = 50

sample_directory = (
    OUTPUT_DIR / "training_samples"
)
sample_directory.mkdir(
    parents=True,
    exist_ok=True,
)

if not np.array_equal(
    df["Transaction ID"].to_numpy(),
    np.arange(len(df), dtype=np.int64),
):
    raise ValueError(
        "Transaction IDs no longer match row positions."
    )

train_mask = df["Split"].eq("train")

positive_train_ids = df.loc[
    train_mask
    & df["Is Laundering"].eq(1),
    "Transaction ID",
].to_numpy(dtype=np.int64)

negative_train_ids = df.loc[
    train_mask
    & df["Is Laundering"].eq(0),
    "Transaction ID",
].to_numpy(dtype=np.int64)

negative_sample_size = min(
    len(negative_train_ids),
    NEGATIVES_PER_POSITIVE
    * len(positive_train_ids),
)

print("Full training transactions:", f"{train_mask.sum():,}")
print(
    "Full training positives:",
    f"{len(positive_train_ids):,}",
)
print(
    "Full training negatives:",
    f"{len(negative_train_ids):,}",
)
print(
    "Sampled negatives per seed:",
    f"{negative_sample_size:,}",
)

sampling_summaries = []

for seed in SEEDS:
    rng = np.random.default_rng(seed)

    selected_negative_ids = rng.choice(
        negative_train_ids,
        size=negative_sample_size,
        replace=False,
    )

    selected_ids = np.concatenate(
        [
            positive_train_ids,
            selected_negative_ids,
        ]
    )

    selected_ids.sort()

    selected_labels = (
        df.loc[selected_ids, "Is Laundering"]
        .to_numpy(dtype=np.int8)
    )

    selected_timestamps = (
        df.loc[selected_ids, "Timestamp"]
        .to_numpy()
    )

    sample_manifest = pd.DataFrame({
        "Transaction ID": selected_ids,
        "Timestamp": selected_timestamps,
        "Is Laundering": selected_labels,
    })

    sample_path = (
        sample_directory
        / f"training_sample_seed_{seed}.parquet"
    )

    sample_manifest.to_parquet(
        sample_path,
        index=False,
    )

    positive_count = int(
        sample_manifest["Is Laundering"].sum()
    )

    negative_count = int(
        len(sample_manifest) - positive_count
    )

    sampling_summaries.append({
        "seed": seed,
        "transactions": len(sample_manifest),
        "positive_transactions": positive_count,
        "negative_transactions": negative_count,
        "negative_to_positive_ratio": (
            negative_count / positive_count
        ),
        "file": str(sample_path),
    })

sampling_summary = pd.DataFrame(
    sampling_summaries
)

sampling_summary.to_csv(
    OUTPUT_DIR / "training_sampling_summary.csv",
    index=False,
)

sampling_configuration = {
    "seeds": SEEDS,
    "method": (
        "Retain every positive training transaction "
        "and randomly sample negative training "
        "transactions without replacement."
    ),
    "negative_transactions_per_positive": (
        NEGATIVES_PER_POSITIVE
    ),
    "validation_resampled": False,
    "test_resampled": False,
    "same_sample_for_all_models_within_seed": True,
}

with open(
    OUTPUT_DIR / "sampling_configuration.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        sampling_configuration,
        file,
        indent=2,
    )

print("\nTRAINING SAMPLING SUMMARY")
display(sampling_summary)

print(
    "\nValidation transactions remain:",
    f"{df['Split'].eq('validation').sum():,}",
)

print(
    "Test transactions remain:",
    f"{df['Split'].eq('test').sum():,}",
)

print(
    "\nSaved training samples in:",
    sample_directory,
)

## Training-only preprocessing

Fits the scaler and categorical encoder on training data only.


In [ ]:
from pathlib import Path
import gc
import json
import joblib
import numpy as np
import pandas as pd
import sklearn

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)

preprocessor_directory = (
    OUTPUT_DIR / "preprocessing"
)

preprocessor_directory.mkdir(
    parents=True,
    exist_ok=True,
)

train_mask = df["Split"].eq("train")
validation_mask = df["Split"].eq("validation")
test_mask = df["Split"].eq("test")

print(
    "Fitting numerical scaler on",
    f"{train_mask.sum():,}",
    "training transactions...",
)

scaler = StandardScaler()

training_numeric = (
    df.loc[train_mask, numeric_features]
    .to_numpy(dtype=np.float32)
)

scaler.fit(training_numeric)

del training_numeric
gc.collect()

print("Fitting categorical encoder...")

try:
    encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=True,
        dtype=np.float32,
    )
except TypeError:
    encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse=True,
        dtype=np.float32,
    )

training_categorical = df.loc[
    train_mask,
    categorical_features,
]

encoder.fit(training_categorical)

del training_categorical
gc.collect()

encoded_feature_names = (
    encoder.get_feature_names_out(
        categorical_features
    )
    .tolist()
)

processed_feature_names = (
    numeric_features
    + encoded_feature_names
)

joblib.dump(
    scaler,
    preprocessor_directory
    / "numeric_scaler.joblib",
)

joblib.dump(
    encoder,
    preprocessor_directory
    / "categorical_encoder.joblib",
)

pd.DataFrame({
    "processed_feature": processed_feature_names
}).to_csv(
    preprocessor_directory
    / "processed_feature_names.csv",
    index=False,
)

unseen_category_records = []

for column, known_values in zip(
    categorical_features,
    encoder.categories_,
):
    known_values = {
        str(value)
        for value in known_values
    }

    for split_name, split_mask in [
        ("validation", validation_mask),
        ("test", test_mask),
    ]:
        observed_values = {
            str(value)
            for value in (
                df.loc[split_mask, column]
                .dropna()
                .unique()
            )
        }

        unseen_values = sorted(
            observed_values - known_values
        )

        unseen_category_records.append({
            "column": column,
            "split": split_name,
            "unseen_category_count": (
                len(unseen_values)
            ),
            "unseen_categories": (
                ", ".join(unseen_values)
            ),
        })

unseen_categories = pd.DataFrame(
    unseen_category_records
)

unseen_categories.to_csv(
    preprocessor_directory
    / "unseen_categories.csv",
    index=False,
)

preprocessing_metadata = {
    "training_rows_used_to_fit": int(
        train_mask.sum()
    ),
    "numeric_input_features": len(
        numeric_features
    ),
    "categorical_input_features": len(
        categorical_features
    ),
    "encoded_categorical_features": len(
        encoded_feature_names
    ),
    "total_processed_features": len(
        processed_feature_names
    ),
    "scikit_learn_version": (
        sklearn.__version__
    ),
    "scaler": "StandardScaler",
    "encoder": "OneHotEncoder",
    "unknown_category_policy": "ignore",
    "fitted_using_training_partition_only": True,
}

with open(
    preprocessor_directory
    / "preprocessing_metadata.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        preprocessing_metadata,
        file,
        indent=2,
    )

print("\nPREPROCESSING SUMMARY")
print(
    "Training rows used:",
    f"{train_mask.sum():,}",
)
print(
    "Numeric input features:",
    len(numeric_features),
)
print(
    "Categorical input features:",
    len(categorical_features),
)
print(
    "Encoded categorical features:",
    len(encoded_feature_names),
)
print(
    "Total processed features:",
    len(processed_feature_names),
)
print(
    "Scikit-learn version:",
    sklearn.__version__,
)

print("\nENCODER CATEGORIES")

for column, values in zip(
    categorical_features,
    encoder.categories_,
):
    print(
        column,
        ":",
        list(values),
    )

print("\nUNSEEN CATEGORY CHECK")
display(unseen_categories)

print(
    "\nSaved preprocessing files in:",
    preprocessor_directory,
)

## Reusable matrices

Transforms and saves training, validation, and test matrices.


In [ ]:
from pathlib import Path
import gc
import json
import time
import numpy as np
import pandas as pd
import scipy
import scipy.sparse as sparse

matrix_directory = OUTPUT_DIR / "matrices"
matrix_directory.mkdir(
    parents=True,
    exist_ok=True,
)

def transform_transaction_ids(
    transaction_ids,
):
    transaction_ids = np.asarray(
        transaction_ids,
        dtype=np.int64,
    )

    numeric_data = (
        df.iloc[transaction_ids][numeric_features]
        .to_numpy(dtype=np.float32)
    )

    numeric_data = scaler.transform(
        numeric_data
    ).astype(np.float32)

    numeric_matrix = sparse.csr_matrix(
        numeric_data,
        dtype=np.float32,
    )

    del numeric_data
    gc.collect()

    categorical_data = (
        df.iloc[transaction_ids][
            categorical_features
        ]
    )

    categorical_matrix = encoder.transform(
        categorical_data
    ).astype(np.float32)

    del categorical_data
    gc.collect()

    combined_matrix = sparse.hstack(
        [
            numeric_matrix,
            categorical_matrix,
        ],
        format="csr",
        dtype=np.float32,
    )

    combined_matrix.sort_indices()

    if combined_matrix.shape[1] != len(
        processed_feature_names
    ):
        raise ValueError(
            "Unexpected processed feature count."
        )

    if not np.isfinite(
        combined_matrix.data
    ).all():
        raise ValueError(
            "The transformed matrix contains "
            "non-finite values."
        )

    return combined_matrix

matrix_summaries = []

def save_matrix_partition(
    name,
    transaction_ids,
    labels,
):
    print("\nTransforming:", name)

    start_time = time.perf_counter()

    matrix = transform_transaction_ids(
        transaction_ids
    )

    elapsed_seconds = (
        time.perf_counter() - start_time
    )

    matrix_path = (
        matrix_directory / f"{name}.npz"
    )

    label_path = (
        matrix_directory / f"{name}_labels.npy"
    )

    id_path = (
        matrix_directory
        / f"{name}_transaction_ids.npy"
    )

    sparse.save_npz(
        matrix_path,
        matrix,
        compressed=True,
    )

    np.save(
        label_path,
        np.asarray(labels, dtype=np.int8),
    )

    np.save(
        id_path,
        np.asarray(
            transaction_ids,
            dtype=np.int64,
        ),
    )

    summary = {
        "partition": name,
        "rows": int(matrix.shape[0]),
        "columns": int(matrix.shape[1]),
        "nonzero_values": int(matrix.nnz),
        "transform_seconds": elapsed_seconds,
        "matrix_size_gb": (
            matrix_path.stat().st_size
            / (1024 ** 3)
        ),
    }

    matrix_summaries.append(summary)

    print(
        "Shape:",
        matrix.shape,
    )
    print(
        "Nonzero values:",
        f"{matrix.nnz:,}",
    )
    print(
        "Transform seconds:",
        f"{elapsed_seconds:.2f}",
    )
    print(
        "Saved matrix size in GB:",
        f"{summary['matrix_size_gb']:.3f}",
    )

    del matrix
    gc.collect()

for seed in SEEDS:
    sample_path = (
        sample_directory
        / f"training_sample_seed_{seed}.parquet"
    )

    sample_manifest = pd.read_parquet(
        sample_path
    )

    sample_ids = sample_manifest[
        "Transaction ID"
    ].to_numpy(dtype=np.int64)

    sample_labels = sample_manifest[
        "Is Laundering"
    ].to_numpy(dtype=np.int8)

    expected_labels = (
        df.iloc[sample_ids]["Is Laundering"]
        .to_numpy(dtype=np.int8)
    )

    if not np.array_equal(
        sample_labels,
        expected_labels,
    ):
        raise ValueError(
            f"Training labels do not match "
            f"for seed {seed}."
        )

    save_matrix_partition(
        f"train_seed_{seed}",
        sample_ids,
        sample_labels,
    )

    del sample_manifest
    del sample_ids
    del sample_labels
    del expected_labels
    gc.collect()

validation_ids = df.loc[
    validation_mask,
    "Transaction ID",
].to_numpy(dtype=np.int64)

validation_labels = df.loc[
    validation_mask,
    "Is Laundering",
].to_numpy(dtype=np.int8)

save_matrix_partition(
    "validation",
    validation_ids,
    validation_labels,
)

del validation_ids
del validation_labels
gc.collect()

test_ids = df.loc[
    test_mask,
    "Transaction ID",
].to_numpy(dtype=np.int64)

test_labels = df.loc[
    test_mask,
    "Is Laundering",
].to_numpy(dtype=np.int8)

save_matrix_partition(
    "test",
    test_ids,
    test_labels,
)

del test_ids
del test_labels
gc.collect()

matrix_summary = pd.DataFrame(
    matrix_summaries
)

matrix_summary.to_csv(
    matrix_directory
    / "matrix_summary.csv",
    index=False,
)

matrix_metadata = {
    "matrix_format": "scipy_csr",
    "matrix_dtype": "float32",
    "processed_feature_count": len(
        processed_feature_names
    ),
    "scipy_version": scipy.__version__,
    "preprocessing_fitted_on_training_only": True,
}

with open(
    matrix_directory / "matrix_metadata.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        matrix_metadata,
        file,
        indent=2,
    )

print("\nMATRIX SUMMARY")
display(matrix_summary)

print(
    "\nAll reusable matrices were saved in:",
    matrix_directory,
)

## Shared evaluation utilities

Defines metrics, thresholds, monitoring, and environment reporting.


In [ ]:
from pathlib import Path
import json
import os
import platform
import subprocess
import sys
import threading
import time

import joblib
import numpy as np
import pandas as pd
import psutil
import scipy
import scipy.sparse as sparse
import sklearn

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)

FIXED_INFERENCE_SIZE = 100_000

def select_validation_threshold(
    labels,
    scores,
):
    precision_values, recall_values, thresholds = (
        precision_recall_curve(
            labels,
            scores,
        )
    )

    if len(thresholds) == 0:
        raise ValueError(
            "Threshold selection failed."
        )

    numerator = (
        2
        * precision_values[:-1]
        * recall_values[:-1]
    )

    denominator = (
        precision_values[:-1]
        + recall_values[:-1]
    )

    f1_values = np.divide(
        numerator,
        denominator,
        out=np.zeros_like(numerator),
        where=denominator > 0,
    )

    best_index = int(
        np.nanargmax(f1_values)
    )

    return {
        "threshold": float(
            thresholds[best_index]
        ),
        "validation_f1": float(
            f1_values[best_index]
        ),
        "validation_precision": float(
            precision_values[best_index]
        ),
        "validation_recall": float(
            recall_values[best_index]
        ),
    }

def calculate_metrics(
    labels,
    scores,
    threshold,
):
    predictions = (
        np.asarray(scores) >= threshold
    ).astype(np.int8)

    tn, fp, fn, tp = confusion_matrix(
        labels,
        predictions,
        labels=[0, 1],
    ).ravel()

    return {
        "accuracy": float(
            accuracy_score(
                labels,
                predictions,
            )
        ),
        "precision": float(
            precision_score(
                labels,
                predictions,
                zero_division=0,
            )
        ),
        "recall": float(
            recall_score(
                labels,
                predictions,
                zero_division=0,
            )
        ),
        "f1": float(
            f1_score(
                labels,
                predictions,
                zero_division=0,
            )
        ),
        "auprc": float(
            average_precision_score(
                labels,
                scores,
            )
        ),
        "roc_auc": float(
            roc_auc_score(
                labels,
                scores,
            )
        ),
        "true_negatives": int(tn),
        "false_positives": int(fp),
        "false_negatives": int(fn),
        "true_positives": int(tp),
        "threshold": float(threshold),
    }

class PeakMemoryMonitor:
    def __init__(self, interval_seconds=0.05):
        self.interval_seconds = interval_seconds
        self.process = psutil.Process(
            os.getpid()
        )
        self.peak_bytes = 0
        self._stop_event = threading.Event()
        self._thread = None

    def _sample(self):
        while not self._stop_event.is_set():
            current_bytes = (
                self.process.memory_info().rss
            )
            self.peak_bytes = max(
                self.peak_bytes,
                current_bytes,
            )
            self._stop_event.wait(
                self.interval_seconds
            )

    def __enter__(self):
        self.peak_bytes = (
            self.process.memory_info().rss
        )
        self._thread = threading.Thread(
            target=self._sample,
            daemon=True,
        )
        self._thread.start()
        return self

    def __exit__(
        self,
        exception_type,
        exception_value,
        traceback,
    ):
        self._stop_event.set()
        self._thread.join()
        current_bytes = (
            self.process.memory_info().rss
        )
        self.peak_bytes = max(
            self.peak_bytes,
            current_bytes,
        )

    @property
    def peak_gb(self):
        return (
            self.peak_bytes
            / (1024 ** 3)
        )

def get_cpu_name():
    try:
        with open(
            "/proc/cpuinfo",
            "r",
            encoding="utf-8",
        ) as file:
            for line in file:
                if line.lower().startswith(
                    "model name"
                ):
                    return (
                        line.split(":", 1)[1]
                        .strip()
                    )
    except Exception:
        pass

    return (
        platform.processor()
        or "Unavailable"
    )

def get_gpu_information():
    try:
        result = subprocess.run(
            [
                "nvidia-smi",
                "--query-gpu=name,memory.total",
                "--format=csv,noheader",
            ],
            capture_output=True,
            text=True,
            check=True,
        )

        output = result.stdout.strip()

        if output:
            return output

    except Exception:
        pass

    return "No NVIDIA GPU detected"

environment = {
    "operating_system": platform.platform(),
    "python_version": sys.version,
    "cpu_model": get_cpu_name(),
    "ram_gb": round(
        psutil.virtual_memory().total
        / (1024 ** 3),
        2,
    ),
    "gpu": get_gpu_information(),
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "scipy_version": scipy.__version__,
    "scikit_learn_version": (
        sklearn.__version__
    ),
    "joblib_version": joblib.__version__,
    "fixed_inference_transaction_count": (
        FIXED_INFERENCE_SIZE
    ),
    "auprc_definition": (
        "Average precision computed from "
        "prediction scores."
    ),
    "threshold_selection": (
        "Maximum suspicious-class F1 "
        "on validation data."
    ),
}

with open(
    OUTPUT_DIR / "software_hardware_environment.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        environment,
        file,
        indent=2,
    )

print("Loading validation and test matrices...")

X_validation = sparse.load_npz(
    matrix_directory / "validation.npz"
)

y_validation = np.load(
    matrix_directory
    / "validation_labels.npy"
)

validation_transaction_ids = np.load(
    matrix_directory
    / "validation_transaction_ids.npy"
)

X_test = sparse.load_npz(
    matrix_directory / "test.npz"
)

y_test = np.load(
    matrix_directory / "test_labels.npy"
)

test_transaction_ids = np.load(
    matrix_directory
    / "test_transaction_ids.npy"
)

test_timestamps = (
    df.iloc[test_transaction_ids]["Timestamp"]
    .to_numpy()
)

dense_test_mask = (
    test_timestamps
    < np.datetime64("2022-09-11")
)

tail_test_mask = ~dense_test_mask

if X_validation.shape[0] != len(
    y_validation
):
    raise ValueError(
        "Validation matrix and labels do not match."
    )

if X_test.shape[0] != len(y_test):
    raise ValueError(
        "Test matrix and labels do not match."
    )

print("\nENVIRONMENT SUMMARY")
print("CPU:", environment["cpu_model"])
print("RAM in GB:", environment["ram_gb"])
print("GPU:", environment["gpu"])
print(
    "Python:",
    platform.python_version(),
)
print(
    "Scikit-learn:",
    sklearn.__version__,
)

print("\nEVALUATION DATA")
print(
    "Validation transactions:",
    f"{len(y_validation):,}",
)
print(
    "Validation positives:",
    f"{int(y_validation.sum()):,}",
)
print(
    "Test transactions:",
    f"{len(y_test):,}",
)
print(
    "Test positives:",
    f"{int(y_test.sum()):,}",
)
print(
    "Dense-period test transactions:",
    f"{int(dense_test_mask.sum()):,}",
)
print(
    "Timestamp-tail test transactions:",
    f"{int(tail_test_mask.sum()):,}",
)

print(
    "\nShared evaluation functions are ready."
)

## Random Forest validation search

Selects Random Forest settings using validation data.


In [ ]:
from pathlib import Path
import gc
import json
import time

import joblib
import numpy as np
import pandas as pd
import scipy.sparse as sparse

from sklearn.ensemble import (
    RandomForestClassifier,
)

rf_directory = (
    OUTPUT_DIR / "models" / "random_forest"
)

rf_directory.mkdir(
    parents=True,
    exist_ok=True,
)

print("Loading seed 42 training matrix...")

X_train_42 = sparse.load_npz(
    matrix_directory / "train_seed_42.npz"
)

y_train_42 = np.load(
    matrix_directory
    / "train_seed_42_labels.npy"
)

rf_candidates = [
    {
        "n_estimators": 250,
        "max_depth": 12,
        "min_samples_leaf": 2,
        "max_features": "sqrt",
    },
    {
        "n_estimators": 250,
        "max_depth": 20,
        "min_samples_leaf": 2,
        "max_features": "sqrt",
    },
    {
        "n_estimators": 250,
        "max_depth": None,
        "min_samples_leaf": 2,
        "max_features": "sqrt",
    },
    {
        "n_estimators": 250,
        "max_depth": 20,
        "min_samples_leaf": 10,
        "max_features": 0.5,
    },
]

rf_search_records = []

search_start = time.perf_counter()

for candidate_number, parameters in enumerate(
    rf_candidates,
    start=1,
):
    print(
        f"\nTraining candidate "
        f"{candidate_number} of "
        f"{len(rf_candidates)}"
    )

    print(parameters)

    model = RandomForestClassifier(
        **parameters,
        criterion="gini",
        bootstrap=True,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1,
    )

    with PeakMemoryMonitor() as memory_monitor:
        training_start = time.perf_counter()

        model.fit(
            X_train_42,
            y_train_42,
        )

        training_seconds = (
            time.perf_counter()
            - training_start
        )

        validation_start = time.perf_counter()

        validation_scores = (
            model.predict_proba(
                X_validation
            )[:, 1]
        )

        validation_inference_seconds = (
            time.perf_counter()
            - validation_start
        )

    threshold_result = (
        select_validation_threshold(
            y_validation,
            validation_scores,
        )
    )

    validation_metrics = calculate_metrics(
        y_validation,
        validation_scores,
        threshold_result["threshold"],
    )

    record = {
        "candidate": candidate_number,
        "n_estimators": parameters[
            "n_estimators"
        ],
        "max_depth": (
            "None"
            if parameters["max_depth"] is None
            else parameters["max_depth"]
        ),
        "min_samples_leaf": parameters[
            "min_samples_leaf"
        ],
        "max_features": parameters[
            "max_features"
        ],
        "validation_accuracy": (
            validation_metrics["accuracy"]
        ),
        "validation_precision": (
            validation_metrics["precision"]
        ),
        "validation_recall": (
            validation_metrics["recall"]
        ),
        "validation_f1": (
            validation_metrics["f1"]
        ),
        "validation_auprc": (
            validation_metrics["auprc"]
        ),
        "validation_roc_auc": (
            validation_metrics["roc_auc"]
        ),
        "selected_threshold": (
            threshold_result["threshold"]
        ),
        "training_seconds": training_seconds,
        "validation_inference_seconds": (
            validation_inference_seconds
        ),
        "peak_cpu_memory_gb": (
            memory_monitor.peak_gb
        ),
    }

    rf_search_records.append(record)

    print(
        "Validation AUPRC:",
        f"{record['validation_auprc']:.6f}",
    )
    print(
        "Validation F1:",
        f"{record['validation_f1']:.6f}",
    )
    print(
        "Selected threshold:",
        f"{record['selected_threshold']:.6f}",
    )
    print(
        "Training seconds:",
        f"{training_seconds:.2f}",
    )

    del model
    del validation_scores
    gc.collect()

rf_search_seconds = (
    time.perf_counter() - search_start
)

rf_search_results = pd.DataFrame(
    rf_search_records
)

rf_search_results.to_csv(
    rf_directory
    / "hyperparameter_search.csv",
    index=False,
)

best_search_row = (
    rf_search_results
    .sort_values(
        [
            "validation_auprc",
            "validation_f1",
        ],
        ascending=False,
    )
    .iloc[0]
)

best_candidate_number = int(
    best_search_row["candidate"]
)

best_rf_parameters = rf_candidates[
    best_candidate_number - 1
].copy()

best_rf_configuration = {
    "model": "Random Forest",
    "selection_metric": "validation_auprc",
    "search_seed": 42,
    "search_seconds": rf_search_seconds,
    "class_weight": "balanced_subsample",
    "best_candidate": best_candidate_number,
    "best_parameters": best_rf_parameters,
    "validation_auprc": float(
        best_search_row[
            "validation_auprc"
        ]
    ),
    "validation_f1": float(
        best_search_row[
            "validation_f1"
        ]
    ),
    "validation_selected_threshold": float(
        best_search_row[
            "selected_threshold"
        ]
    ),
}

with open(
    rf_directory
    / "selected_hyperparameters.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        best_rf_configuration,
        file,
        indent=2,
    )

print("\nRANDOM FOREST SEARCH RESULTS")
display(rf_search_results)

print(
    "\nBest candidate:",
    best_candidate_number,
)

print(
    "Best parameters:",
    best_rf_parameters,
)

print(
    "Total search seconds:",
    f"{rf_search_seconds:.2f}",
)

del X_train_42
del y_train_42
gc.collect()

## Random Forest final seed runs

Trains and records all Random Forest seeds.


In [ ]:
from pathlib import Path
import gc
import json
import time

import joblib
import numpy as np
import pandas as pd
import scipy.sparse as sparse

from sklearn.ensemble import (
    RandomForestClassifier,
)

with open(
    rf_directory
    / "selected_hyperparameters.json",
    "r",
    encoding="utf-8",
) as file:
    saved_rf_configuration = json.load(file)

best_rf_parameters = (
    saved_rf_configuration[
        "best_parameters"
    ]
)

rf_prediction_directory = (
    OUTPUT_DIR
    / "predictions"
    / "random_forest"
)

rf_prediction_directory.mkdir(
    parents=True,
    exist_ok=True,
)

rf_seed_result_directory = (
    rf_directory / "seed_results"
)

rf_seed_result_directory.mkdir(
    parents=True,
    exist_ok=True,
)

rf_results_path = (
    rf_directory / "metrics_by_seed.csv"
)

rf_result_records = []

def add_metric_prefix(
    destination,
    prefix,
    metrics,
):
    for key, value in metrics.items():
        destination[
            f"{prefix}_{key}"
        ] = value

for seed in SEEDS:
    print(
        f"\nRandom Forest seed {seed}"
    )

    model_path = (
        rf_directory
        / f"random_forest_seed_{seed}.joblib"
    )

    prediction_path = (
        rf_prediction_directory
        / f"test_predictions_seed_{seed}.parquet"
    )

    seed_result_path = (
        rf_seed_result_directory
        / f"seed_{seed}.json"
    )

    if (
        model_path.exists()
        and prediction_path.exists()
        and seed_result_path.exists()
    ):
        print(
            "Completed checkpoint found. "
            "Skipping this seed."
        )

        with open(
            seed_result_path,
            "r",
            encoding="utf-8",
        ) as file:
            existing_record = json.load(file)

        rf_result_records.append(
            existing_record
        )

        continue

    X_train = sparse.load_npz(
        matrix_directory
        / f"train_seed_{seed}.npz"
    )

    y_train = np.load(
        matrix_directory
        / f"train_seed_{seed}_labels.npy"
    )

    model = RandomForestClassifier(
        **best_rf_parameters,
        criterion="gini",
        bootstrap=True,
        class_weight="balanced_subsample",
        random_state=seed,
        n_jobs=-1,
    )

    with PeakMemoryMonitor() as memory_monitor:
        training_start = time.perf_counter()

        model.fit(
            X_train,
            y_train,
        )

        training_seconds = (
            time.perf_counter()
            - training_start
        )

        validation_start = time.perf_counter()

        validation_scores = (
            model.predict_proba(
                X_validation
            )[:, 1]
        )

        validation_inference_seconds = (
            time.perf_counter()
            - validation_start
        )

        threshold_result = (
            select_validation_threshold(
                y_validation,
                validation_scores,
            )
        )

        validation_metrics = (
            calculate_metrics(
                y_validation,
                validation_scores,
                threshold_result[
                    "threshold"
                ],
            )
        )

        fixed_size = min(
            FIXED_INFERENCE_SIZE,
            X_test.shape[0],
        )

        fixed_inference_start = (
            time.perf_counter()
        )

        fixed_scores = (
            model.predict_proba(
                X_test[:fixed_size]
            )[:, 1]
        )

        fixed_inference_seconds = (
            time.perf_counter()
            - fixed_inference_start
        )

        remaining_start = (
            time.perf_counter()
        )

        remaining_scores = (
            model.predict_proba(
                X_test[fixed_size:]
            )[:, 1]
        )

        remaining_inference_seconds = (
            time.perf_counter()
            - remaining_start
        )

        test_scores = np.concatenate(
            [
                fixed_scores,
                remaining_scores,
            ]
        )

    full_test_inference_seconds = (
        fixed_inference_seconds
        + remaining_inference_seconds
    )

    selected_threshold = (
        threshold_result["threshold"]
    )

    test_metrics = calculate_metrics(
        y_test,
        test_scores,
        selected_threshold,
    )

    dense_test_metrics = calculate_metrics(
        y_test[dense_test_mask],
        test_scores[dense_test_mask],
        selected_threshold,
    )

    tail_test_metrics = calculate_metrics(
        y_test[tail_test_mask],
        test_scores[tail_test_mask],
        selected_threshold,
    )

    test_predictions = (
        test_scores >= selected_threshold
    ).astype(np.int8)

    prediction_table = pd.DataFrame({
        "Transaction ID": (
            test_transaction_ids
        ),
        "true_label": y_test,
        "prediction_score": (
            test_scores.astype(np.float32)
        ),
        "predicted_label": (
            test_predictions
        ),
        "test_period": np.where(
            dense_test_mask,
            "dense_period",
            "timestamp_tail",
        ),
    })

    prediction_table.to_parquet(
        prediction_path,
        index=False,
    )

    serialization_start = (
        time.perf_counter()
    )

    joblib.dump(
        model,
        model_path,
        compress=3,
    )

    serialization_seconds = (
        time.perf_counter()
        - serialization_start
    )

    record = {
        "model": "Random Forest",
        "seed": int(seed),
        "training_transactions": int(
            len(y_train)
        ),
        "training_positives": int(
            y_train.sum()
        ),
        "training_seconds": float(
            training_seconds
        ),
        "validation_inference_seconds": float(
            validation_inference_seconds
        ),
        "fixed_inference_transactions": int(
            fixed_size
        ),
        "fixed_inference_seconds": float(
            fixed_inference_seconds
        ),
        "full_test_inference_seconds": float(
            full_test_inference_seconds
        ),
        "serialization_seconds": float(
            serialization_seconds
        ),
        "peak_cpu_memory_gb": float(
            memory_monitor.peak_gb
        ),
        "peak_gpu_memory_gb": 0.0,
        "validation_selected_threshold": float(
            selected_threshold
        ),
    }

    add_metric_prefix(
        record,
        "validation",
        validation_metrics,
    )

    add_metric_prefix(
        record,
        "test",
        test_metrics,
    )

    add_metric_prefix(
        record,
        "dense_test",
        dense_test_metrics,
    )

    add_metric_prefix(
        record,
        "tail_test",
        tail_test_metrics,
    )

    with open(
        seed_result_path,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            record,
            file,
            indent=2,
        )

    rf_result_records.append(record)

    pd.DataFrame(
        rf_result_records
    ).sort_values(
        "seed"
    ).to_csv(
        rf_results_path,
        index=False,
    )

    print(
        "Selected threshold:",
        f"{selected_threshold:.6f}",
    )
    print(
        "Test precision:",
        f"{test_metrics['precision']:.6f}",
    )
    print(
        "Test recall:",
        f"{test_metrics['recall']:.6f}",
    )
    print(
        "Test F1:",
        f"{test_metrics['f1']:.6f}",
    )
    print(
        "Test AUPRC:",
        f"{test_metrics['auprc']:.6f}",
    )
    print(
        "Test ROC-AUC:",
        f"{test_metrics['roc_auc']:.6f}",
    )
    print(
        "Training seconds:",
        f"{training_seconds:.2f}",
    )
    print(
        "Checkpoint saved."
    )

    del X_train
    del y_train
    del model
    del validation_scores
    del fixed_scores
    del remaining_scores
    del test_scores
    del test_predictions
    del prediction_table
    gc.collect()

rf_results = (
    pd.DataFrame(rf_result_records)
    .sort_values("seed")
    .reset_index(drop=True)
)

rf_results.to_csv(
    rf_results_path,
    index=False,
)

display_columns = [
    "seed",
    "validation_selected_threshold",
    "test_precision",
    "test_recall",
    "test_f1",
    "test_auprc",
    "test_roc_auc",
    "test_true_negatives",
    "test_false_positives",
    "test_false_negatives",
    "test_true_positives",
    "training_seconds",
    "fixed_inference_seconds",
    "peak_cpu_memory_gb",
]

print("\nRANDOM FOREST RESULTS")
display(
    rf_results[display_columns]
)

print(
    "\nSaved results:",
    rf_results_path,
)

## XGBoost and Linear SVM experiments

Runs validation search and final seeds for both models.


In [ ]:
from pathlib import Path
import gc
import json
import os
import subprocess
import threading
import time

import joblib
import numpy as np
import pandas as pd
import scipy.sparse as sparse
import xgboost

from sklearn.svm import LinearSVC


# ============================================================
# GPU memory monitoring
# ============================================================

class GpuMemoryMonitor:
    def __init__(
        self,
        enabled=True,
        interval_seconds=0.10,
    ):
        self.enabled = enabled
        self.interval_seconds = interval_seconds
        self.peak_mb = 0.0
        self._stop_event = threading.Event()
        self._thread = None

    def _read_current_process_memory(self):
        if not self.enabled:
            return 0.0

        try:
            result = subprocess.run(
                [
                    "nvidia-smi",
                    "--query-compute-apps="
                    "pid,used_gpu_memory",
                    "--format=csv,noheader,nounits",
                ],
                capture_output=True,
                text=True,
                check=True,
                timeout=3,
            )

            current_pid = os.getpid()
            total_mb = 0.0

            for line in result.stdout.splitlines():
                parts = [
                    part.strip()
                    for part in line.split(",")
                ]

                if len(parts) != 2:
                    continue

                try:
                    process_id = int(parts[0])
                    memory_mb = float(parts[1])
                except ValueError:
                    continue

                if process_id == current_pid:
                    total_mb += memory_mb

            return total_mb

        except Exception:
            return 0.0

    def _sample(self):
        while not self._stop_event.is_set():
            self.peak_mb = max(
                self.peak_mb,
                self._read_current_process_memory(),
            )

            self._stop_event.wait(
                self.interval_seconds
            )

    def __enter__(self):
        if self.enabled:
            self.peak_mb = (
                self._read_current_process_memory()
            )

            self._thread = threading.Thread(
                target=self._sample,
                daemon=True,
            )

            self._thread.start()

        return self

    def __exit__(
        self,
        exception_type,
        exception_value,
        traceback,
    ):
        if self.enabled and self._thread is not None:
            self._stop_event.set()
            self._thread.join()

            self.peak_mb = max(
                self.peak_mb,
                self._read_current_process_memory(),
            )

    @property
    def peak_gb(self):
        return self.peak_mb / 1024


# ============================================================
# Shared model functions
# ============================================================

def probability_scores(model, matrix):
    return model.predict_proba(matrix)[:, 1]


def decision_scores(model, matrix):
    return model.decision_function(matrix)


def ordinary_fit(model, matrix, labels):
    model.fit(matrix, labels)


def save_xgboost_model(model, path):
    model.save_model(str(path))


def save_joblib_model(model, path):
    joblib.dump(
        model,
        path,
        compress=3,
    )


def run_validation_search(
    model_key,
    model_label,
    model_directory,
    candidates,
    model_builder,
    fit_function,
    score_function,
    uses_gpu,
):
    model_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    selected_path = (
        model_directory
        / "selected_hyperparameters.json"
    )

    search_result_path = (
        model_directory
        / "hyperparameter_search.csv"
    )

    search_space_path = (
        model_directory
        / "hyperparameter_search_space.json"
    )

    if (
        selected_path.exists()
        and search_result_path.exists()
    ):
        print(
            f"\nExisting completed search found "
            f"for {model_label}."
        )

        with open(
            selected_path,
            "r",
            encoding="utf-8",
        ) as file:
            saved_selection = json.load(file)

        existing_results = pd.read_csv(
            search_result_path
        )

        display(existing_results)

        return saved_selection[
            "best_parameters"
        ]

    with open(
        search_space_path,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            {
                "model": model_label,
                "selection_metric": (
                    "validation_auprc"
                ),
                "search_seed": 42,
                "candidates": candidates,
            },
            file,
            indent=2,
        )

    X_train_search = sparse.load_npz(
        matrix_directory
        / "train_seed_42.npz"
    )

    y_train_search = np.load(
        matrix_directory
        / "train_seed_42_labels.npy"
    )

    search_records = []
    complete_search_start = time.perf_counter()

    for candidate_number, parameters in enumerate(
        candidates,
        start=1,
    ):
        print(
            f"\n{model_label} candidate "
            f"{candidate_number} of "
            f"{len(candidates)}"
        )

        print(parameters)

        model = model_builder(
            parameters,
            42,
            y_train_search,
        )

        with (
            PeakMemoryMonitor()
            as cpu_memory_monitor,
            GpuMemoryMonitor(
                enabled=uses_gpu
            )
            as gpu_memory_monitor,
        ):
            training_start = time.perf_counter()

            fit_function(
                model,
                X_train_search,
                y_train_search,
            )

            training_seconds = (
                time.perf_counter()
                - training_start
            )

            validation_start = (
                time.perf_counter()
            )

            validation_scores = (
                score_function(
                    model,
                    X_validation,
                )
            )

            validation_inference_seconds = (
                time.perf_counter()
                - validation_start
            )

        threshold_result = (
            select_validation_threshold(
                y_validation,
                validation_scores,
            )
        )

        validation_metrics = calculate_metrics(
            y_validation,
            validation_scores,
            threshold_result["threshold"],
        )

        record = {
            "candidate": candidate_number,
            "parameters_json": json.dumps(
                parameters,
                sort_keys=True,
            ),
            "validation_accuracy": (
                validation_metrics["accuracy"]
            ),
            "validation_precision": (
                validation_metrics["precision"]
            ),
            "validation_recall": (
                validation_metrics["recall"]
            ),
            "validation_f1": (
                validation_metrics["f1"]
            ),
            "validation_auprc": (
                validation_metrics["auprc"]
            ),
            "validation_roc_auc": (
                validation_metrics["roc_auc"]
            ),
            "selected_threshold": (
                threshold_result["threshold"]
            ),
            "training_seconds": (
                training_seconds
            ),
            "validation_inference_seconds": (
                validation_inference_seconds
            ),
            "peak_cpu_memory_gb": (
                cpu_memory_monitor.peak_gb
            ),
            "peak_gpu_memory_gb": (
                gpu_memory_monitor.peak_gb
            ),
        }

        search_records.append(record)

        pd.DataFrame(
            search_records
        ).to_csv(
            search_result_path,
            index=False,
        )

        print(
            "Validation AUPRC:",
            f"{record['validation_auprc']:.6f}",
        )

        print(
            "Validation F1:",
            f"{record['validation_f1']:.6f}",
        )

        print(
            "Selected threshold:",
            f"{record['selected_threshold']:.6f}",
        )

        print(
            "Training seconds:",
            f"{training_seconds:.2f}",
        )

        del model
        del validation_scores
        gc.collect()

    total_search_seconds = (
        time.perf_counter()
        - complete_search_start
    )

    search_results = pd.DataFrame(
        search_records
    )

    best_row = (
        search_results
        .sort_values(
            [
                "validation_auprc",
                "validation_f1",
            ],
            ascending=False,
        )
        .iloc[0]
    )

    best_candidate = int(
        best_row["candidate"]
    )

    best_parameters = candidates[
        best_candidate - 1
    ].copy()

    selection = {
        "model": model_label,
        "selection_metric": (
            "validation_auprc"
        ),
        "search_seed": 42,
        "search_seconds": (
            total_search_seconds
        ),
        "best_candidate": best_candidate,
        "best_parameters": best_parameters,
        "validation_auprc": float(
            best_row["validation_auprc"]
        ),
        "validation_f1": float(
            best_row["validation_f1"]
        ),
        "validation_selected_threshold": float(
            best_row["selected_threshold"]
        ),
    }

    with open(
        selected_path,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            selection,
            file,
            indent=2,
        )

    print(
        f"\n{model_label.upper()} "
        f"SEARCH RESULTS"
    )

    display(search_results)

    print(
        "Best candidate:",
        best_candidate,
    )

    print(
        "Best parameters:",
        best_parameters,
    )

    print(
        "Total search seconds:",
        f"{total_search_seconds:.2f}",
    )

    del X_train_search
    del y_train_search
    gc.collect()

    return best_parameters


def run_all_model_seeds(
    model_key,
    model_label,
    model_directory,
    selected_parameters,
    model_builder,
    fit_function,
    score_function,
    save_model_function,
    model_extension,
    uses_gpu,
):
    prediction_directory = (
        OUTPUT_DIR
        / "predictions"
        / model_key
    )

    prediction_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    seed_result_directory = (
        model_directory / "seed_results"
    )

    seed_result_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    result_csv_path = (
        model_directory
        / "metrics_by_seed.csv"
    )

    result_records = []

    for seed in SEEDS:
        print(
            f"\n{model_label} seed {seed}"
        )

        model_path = (
            model_directory
            / f"{model_key}_seed_{seed}"
            f"{model_extension}"
        )

        prediction_path = (
            prediction_directory
            / f"test_predictions_seed_"
            f"{seed}.parquet"
        )

        seed_result_path = (
            seed_result_directory
            / f"seed_{seed}.json"
        )

        if (
            model_path.exists()
            and prediction_path.exists()
            and seed_result_path.exists()
        ):
            print(
                "Completed checkpoint found. "
                "Skipping this seed."
            )

            with open(
                seed_result_path,
                "r",
                encoding="utf-8",
            ) as file:
                existing_record = json.load(file)

            result_records.append(
                existing_record
            )

            continue

        X_train = sparse.load_npz(
            matrix_directory
            / f"train_seed_{seed}.npz"
        )

        y_train = np.load(
            matrix_directory
            / f"train_seed_{seed}_labels.npy"
        )

        model = model_builder(
            selected_parameters,
            seed,
            y_train,
        )

        with (
            PeakMemoryMonitor()
            as cpu_memory_monitor,
            GpuMemoryMonitor(
                enabled=uses_gpu
            )
            as gpu_memory_monitor,
        ):
            training_start = time.perf_counter()

            fit_function(
                model,
                X_train,
                y_train,
            )

            training_seconds = (
                time.perf_counter()
                - training_start
            )

            validation_start = (
                time.perf_counter()
            )

            validation_scores = (
                score_function(
                    model,
                    X_validation,
                )
            )

            validation_inference_seconds = (
                time.perf_counter()
                - validation_start
            )

            threshold_result = (
                select_validation_threshold(
                    y_validation,
                    validation_scores,
                )
            )

            validation_metrics = (
                calculate_metrics(
                    y_validation,
                    validation_scores,
                    threshold_result[
                        "threshold"
                    ],
                )
            )

            fixed_size = min(
                FIXED_INFERENCE_SIZE,
                X_test.shape[0],
            )

            fixed_inference_start = (
                time.perf_counter()
            )

            fixed_scores = score_function(
                model,
                X_test[:fixed_size],
            )

            fixed_inference_seconds = (
                time.perf_counter()
                - fixed_inference_start
            )

            remaining_start = (
                time.perf_counter()
            )

            remaining_scores = score_function(
                model,
                X_test[fixed_size:],
            )

            remaining_inference_seconds = (
                time.perf_counter()
                - remaining_start
            )

            test_scores = np.concatenate(
                [
                    fixed_scores,
                    remaining_scores,
                ]
            )

        full_test_inference_seconds = (
            fixed_inference_seconds
            + remaining_inference_seconds
        )

        selected_threshold = (
            threshold_result["threshold"]
        )

        test_metrics = calculate_metrics(
            y_test,
            test_scores,
            selected_threshold,
        )

        dense_test_metrics = calculate_metrics(
            y_test[dense_test_mask],
            test_scores[dense_test_mask],
            selected_threshold,
        )

        tail_test_metrics = calculate_metrics(
            y_test[tail_test_mask],
            test_scores[tail_test_mask],
            selected_threshold,
        )

        test_predictions = (
            test_scores >= selected_threshold
        ).astype(np.int8)

        prediction_table = pd.DataFrame({
            "Transaction ID": (
                test_transaction_ids
            ),
            "true_label": y_test,
            "prediction_score": (
                np.asarray(
                    test_scores,
                    dtype=np.float32,
                )
            ),
            "predicted_label": (
                test_predictions
            ),
            "test_period": np.where(
                dense_test_mask,
                "dense_period",
                "timestamp_tail",
            ),
        })

        prediction_table.to_parquet(
            prediction_path,
            index=False,
        )

        serialization_start = (
            time.perf_counter()
        )

        save_model_function(
            model,
            model_path,
        )

        serialization_seconds = (
            time.perf_counter()
            - serialization_start
        )

        record = {
            "model": model_label,
            "seed": int(seed),
            "training_transactions": int(
                len(y_train)
            ),
            "training_positives": int(
                y_train.sum()
            ),
            "training_seconds": float(
                training_seconds
            ),
            "validation_inference_seconds": float(
                validation_inference_seconds
            ),
            "fixed_inference_transactions": int(
                fixed_size
            ),
            "fixed_inference_seconds": float(
                fixed_inference_seconds
            ),
            "full_test_inference_seconds": float(
                full_test_inference_seconds
            ),
            "serialization_seconds": float(
                serialization_seconds
            ),
            "peak_cpu_memory_gb": float(
                cpu_memory_monitor.peak_gb
            ),
            "peak_gpu_memory_gb": float(
                gpu_memory_monitor.peak_gb
            ),
            "validation_selected_threshold": float(
                selected_threshold
            ),
        }

        add_metric_prefix(
            record,
            "validation",
            validation_metrics,
        )

        add_metric_prefix(
            record,
            "test",
            test_metrics,
        )

        add_metric_prefix(
            record,
            "dense_test",
            dense_test_metrics,
        )

        add_metric_prefix(
            record,
            "tail_test",
            tail_test_metrics,
        )

        with open(
            seed_result_path,
            "w",
            encoding="utf-8",
        ) as file:
            json.dump(
                record,
                file,
                indent=2,
            )

        result_records.append(record)

        pd.DataFrame(
            result_records
        ).sort_values(
            "seed"
        ).to_csv(
            result_csv_path,
            index=False,
        )

        print(
            "Selected threshold:",
            f"{selected_threshold:.6f}",
        )

        print(
            "Test precision:",
            f"{test_metrics['precision']:.6f}",
        )

        print(
            "Test recall:",
            f"{test_metrics['recall']:.6f}",
        )

        print(
            "Test F1:",
            f"{test_metrics['f1']:.6f}",
        )

        print(
            "Test AUPRC:",
            f"{test_metrics['auprc']:.6f}",
        )

        print(
            "Test ROC-AUC:",
            f"{test_metrics['roc_auc']:.6f}",
        )

        print(
            "Training seconds:",
            f"{training_seconds:.2f}",
        )

        print("Checkpoint saved.")

        del X_train
        del y_train
        del model
        del validation_scores
        del fixed_scores
        del remaining_scores
        del test_scores
        del test_predictions
        del prediction_table
        gc.collect()

    results = (
        pd.DataFrame(result_records)
        .sort_values("seed")
        .reset_index(drop=True)
    )

    results.to_csv(
        result_csv_path,
        index=False,
    )

    display_columns = [
        "seed",
        "validation_selected_threshold",
        "test_precision",
        "test_recall",
        "test_f1",
        "test_auprc",
        "test_roc_auc",
        "test_true_negatives",
        "test_false_positives",
        "test_false_negatives",
        "test_true_positives",
        "training_seconds",
        "fixed_inference_seconds",
        "peak_cpu_memory_gb",
        "peak_gpu_memory_gb",
    ]

    print(
        f"\n{model_label.upper()} RESULTS"
    )

    display(
        results[display_columns]
    )

    return results


# ============================================================
# XGBoost configuration
# ============================================================

xgboost_directory = (
    OUTPUT_DIR / "models" / "xgboost"
)

xgboost_candidates = [
    {
        "n_estimators": 300,
        "max_depth": 6,
        "learning_rate": 0.05,
        "min_child_weight": 1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
    },
    {
        "n_estimators": 300,
        "max_depth": 8,
        "learning_rate": 0.05,
        "min_child_weight": 5,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
    },
    {
        "n_estimators": 500,
        "max_depth": 6,
        "learning_rate": 0.03,
        "min_child_weight": 5,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
    },
]

xgboost_major_version = int(
    xgboost.__version__.split(".")[0]
)

def build_xgboost_model(
    parameters,
    seed,
    training_labels,
):
    positive_count = int(
        training_labels.sum()
    )

    negative_count = int(
        len(training_labels)
        - positive_count
    )

    scale_pos_weight = (
        negative_count / positive_count
    )

    device_parameters = {}

    if xgboost_major_version >= 2:
        device_parameters = {
            "tree_method": "hist",
            "device": "cuda:0",
        }
    else:
        device_parameters = {
            "tree_method": "gpu_hist",
            "predictor": "gpu_predictor",
        }

    return xgboost.XGBClassifier(
        **parameters,
        **device_parameters,
        objective="binary:logistic",
        eval_metric="aucpr",
        scale_pos_weight=scale_pos_weight,
        reg_lambda=1.0,
        reg_alpha=0.0,
        max_bin=256,
        random_state=seed,
        n_jobs=-1,
        verbosity=0,
    )


print(
    "\nXGBoost version:",
    xgboost.__version__,
)

best_xgboost_parameters = (
    run_validation_search(
        model_key="xgboost",
        model_label="XGBoost",
        model_directory=xgboost_directory,
        candidates=xgboost_candidates,
        model_builder=build_xgboost_model,
        fit_function=ordinary_fit,
        score_function=probability_scores,
        uses_gpu=True,
    )
)

xgboost_results = run_all_model_seeds(
    model_key="xgboost",
    model_label="XGBoost",
    model_directory=xgboost_directory,
    selected_parameters=(
        best_xgboost_parameters
    ),
    model_builder=build_xgboost_model,
    fit_function=ordinary_fit,
    score_function=probability_scores,
    save_model_function=(
        save_xgboost_model
    ),
    model_extension=".json",
    uses_gpu=True,
)


# ============================================================
# Linear SVM configuration
# ============================================================

svm_directory = (
    OUTPUT_DIR / "models" / "svm"
)

svm_candidates = [
    {
        "C": 0.01,
        "tol": 0.0001,
        "max_iter": 10000,
    },
    {
        "C": 0.1,
        "tol": 0.0001,
        "max_iter": 10000,
    },
    {
        "C": 1.0,
        "tol": 0.0001,
        "max_iter": 10000,
    },
]

def build_svm_model(
    parameters,
    seed,
    training_labels,
):
    return LinearSVC(
        **parameters,
        class_weight="balanced",
        dual="auto",
        random_state=seed,
    )


best_svm_parameters = (
    run_validation_search(
        model_key="svm",
        model_label="Linear SVM",
        model_directory=svm_directory,
        candidates=svm_candidates,
        model_builder=build_svm_model,
        fit_function=ordinary_fit,
        score_function=decision_scores,
        uses_gpu=False,
    )
)

svm_results = run_all_model_seeds(
    model_key="svm",
    model_label="Linear SVM",
    model_directory=svm_directory,
    selected_parameters=(
        best_svm_parameters
    ),
    model_builder=build_svm_model,
    fit_function=ordinary_fit,
    score_function=decision_scores,
    save_model_function=(
        save_joblib_model
    ),
    model_extension=".joblib",
    uses_gpu=False,
)


# ============================================================
# Combined conventional model checkpoint
# ============================================================

rf_results = pd.read_csv(
    OUTPUT_DIR
    / "models"
    / "random_forest"
    / "metrics_by_seed.csv"
)

conventional_results = pd.concat(
    [
        rf_results,
        xgboost_results,
        svm_results,
    ],
    ignore_index=True,
)

conventional_results.to_csv(
    OUTPUT_DIR
    / "conventional_metrics_by_seed.csv",
    index=False,
)

final_display_columns = [
    "model",
    "seed",
    "validation_selected_threshold",
    "test_precision",
    "test_recall",
    "test_f1",
    "test_auprc",
    "test_roc_auc",
    "test_true_negatives",
    "test_false_positives",
    "test_false_negatives",
    "test_true_positives",
    "training_seconds",
    "fixed_inference_seconds",
    "peak_cpu_memory_gb",
    "peak_gpu_memory_gb",
]

print(
    "\nALL CONVENTIONAL MODEL RESULTS"
)

display(
    conventional_results[
        final_display_columns
    ]
)

print(
    "\nSaved combined results to:",
    OUTPUT_DIR
    / "conventional_metrics_by_seed.csv",
)

## GNN graph construction, search, pilot runs, and ablation

Builds training-only graphs and runs GCN, GraphSAGE, GAT, and the fixed ablation.


In [ ]:
from pathlib import Path
import copy
import gc
import json
import os
import subprocess
import sys
import time

import numpy as np
import pandas as pd
import scipy.sparse as sparse

import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    import torch_geometric
    from torch_geometric.nn import (
        GATConv,
        GCNConv,
        SAGEConv,
    )
except ImportError:
    print("Installing PyTorch Geometric...")

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "torch-geometric==2.7.0",
            "-q",
        ]
    )

    import torch_geometric

    from torch_geometric.nn import (
        GATConv,
        GCNConv,
        SAGEConv,
    )


# ============================================================
# General configuration
# ============================================================

if not torch.cuda.is_available():
    raise RuntimeError(
        "A CUDA GPU is required for this stage."
    )

DEVICE = torch.device("cuda:0")
NUM_NODES = int(len(node_mapping))
EDGE_FEATURE_COUNT = int(
    len(processed_feature_names)
)
NODE_FEATURE_COUNT = (
    2 * EDGE_FEATURE_COUNT
)

GNN_SEARCH_EPOCHS = 4
GNN_FINAL_EPOCHS = 8
GNN_PATIENCE = 2
GNN_SCORE_BATCH_SIZE = 100_000

gnn_root = OUTPUT_DIR / "gnn"
gnn_root.mkdir(parents=True, exist_ok=True)

graph_cache_directory = (
    gnn_root / "graph_cache"
)
graph_cache_directory.mkdir(
    parents=True,
    exist_ok=True,
)

gnn_prediction_directory = (
    OUTPUT_DIR / "predictions" / "gnn"
)
gnn_prediction_directory.mkdir(
    parents=True,
    exist_ok=True,
)

print("PyTorch version:", torch.__version__)
print(
    "PyTorch Geometric version:",
    torch_geometric.__version__,
)
print("CUDA device:", torch.cuda.get_device_name(0))
print("Account nodes:", f"{NUM_NODES:,}")
print("Edge features:", EDGE_FEATURE_COUNT)
print("Node features:", NODE_FEATURE_COUNT)


# ============================================================
# Update environment record
# ============================================================

environment_path = (
    OUTPUT_DIR
    / "software_hardware_environment.json"
)

with open(
    environment_path,
    "r",
    encoding="utf-8",
) as file:
    environment = json.load(file)

environment.update({
    "pytorch_version": torch.__version__,
    "pytorch_geometric_version": (
        torch_geometric.__version__
    ),
    "cuda_runtime_version": torch.version.cuda,
    "cuda_device_used": (
        torch.cuda.get_device_name(0)
    ),
})

with open(
    environment_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        environment,
        file,
        indent=2,
    )


# ============================================================
# Prepare complete training edge features
# ============================================================

full_train_matrix_path = (
    matrix_directory / "train_full.npz"
)

train_transaction_ids = df.loc[
    train_mask,
    "Transaction ID",
].to_numpy(dtype=np.int64)

if not full_train_matrix_path.exists():
    print(
        "\nTransforming the complete training "
        "partition for graph construction..."
    )

    transformation_start = time.perf_counter()

    X_train_full = transform_transaction_ids(
        train_transaction_ids
    )

    sparse.save_npz(
        full_train_matrix_path,
        X_train_full,
        compressed=True,
    )

    full_train_transform_seconds = (
        time.perf_counter()
        - transformation_start
    )

    print(
        "Complete training matrix:",
        X_train_full.shape,
    )

    print(
        "Transformation seconds:",
        f"{full_train_transform_seconds:.2f}",
    )

    del X_train_full
    gc.collect()
else:
    print(
        "\nExisting complete training matrix found."
    )


# ============================================================
# Prepare dense evaluation feature checkpoints
# ============================================================

validation_dense_path = (
    matrix_directory
    / "validation_dense_float32.npy"
)

test_dense_path = (
    matrix_directory
    / "test_dense_float32.npy"
)

if not validation_dense_path.exists():
    print("Saving dense validation features...")

    np.save(
        validation_dense_path,
        X_validation.toarray().astype(
            np.float32
        ),
    )

if not test_dense_path.exists():
    print("Saving dense test features...")

    np.save(
        test_dense_path,
        X_test.toarray().astype(
            np.float32
        ),
    )

validation_dense = np.load(
    validation_dense_path,
    mmap_mode="r",
)

test_dense = np.load(
    test_dense_path,
    mmap_mode="r",
)

validation_sources = df.iloc[
    validation_transaction_ids
]["Source Node ID"].to_numpy(
    dtype=np.int64
)

validation_destinations = df.iloc[
    validation_transaction_ids
]["Destination Node ID"].to_numpy(
    dtype=np.int64
)

test_sources = df.iloc[
    test_transaction_ids
]["Source Node ID"].to_numpy(
    dtype=np.int64
)

test_destinations = df.iloc[
    test_transaction_ids
]["Destination Node ID"].to_numpy(
    dtype=np.int64
)

train_sources_all = df.iloc[
    train_transaction_ids
]["Source Node ID"].to_numpy(
    dtype=np.int64
)

train_destinations_all = df.iloc[
    train_transaction_ids
]["Destination Node ID"].to_numpy(
    dtype=np.int64
)


# ============================================================
# Build a disjoint graph checkpoint for each seed
# ============================================================

def graph_cache_path(seed):
    return (
        graph_cache_directory
        / f"training_graph_seed_{seed}.pt"
    )


def graph_summary_path(seed):
    return (
        graph_cache_directory
        / f"training_graph_seed_{seed}.json"
    )


def build_graph_cache(seed):
    cache_path = graph_cache_path(seed)
    summary_path = graph_summary_path(seed)

    if cache_path.exists() and summary_path.exists():
        print(
            f"Graph checkpoint found for seed {seed}."
        )

        with open(
            summary_path,
            "r",
            encoding="utf-8",
        ) as file:
            return json.load(file)

    print(
        f"\nBuilding training graph for seed {seed}..."
    )

    build_start = time.perf_counter()

    supervision_ids = np.load(
        matrix_directory
        / f"train_seed_{seed}_transaction_ids.npy"
    ).astype(np.int64)

    if supervision_ids.min() < 0:
        raise ValueError(
            "A negative transaction ID was found."
        )

    if supervision_ids.max() >= len(
        train_transaction_ids
    ):
        raise ValueError(
            "A supervision edge is outside "
            "the training partition."
        )

    context_mask = np.ones(
        len(train_transaction_ids),
        dtype=bool,
    )

    context_mask[supervision_ids] = False

    context_positions = np.flatnonzero(
        context_mask
    )

    context_sources = train_sources_all[
        context_positions
    ]

    context_destinations = (
        train_destinations_all[
            context_positions
        ]
    )

    if np.intersect1d(
        context_positions,
        supervision_ids,
        assume_unique=True,
    ).size != 0:
        raise ValueError(
            "Supervision edges remain in "
            "the message-passing graph."
        )

    X_train_full = sparse.load_npz(
        full_train_matrix_path
    )

    X_context = X_train_full[
        context_positions
    ]

    context_edge_count = int(
        len(context_positions)
    )

    context_column_ids = np.arange(
        context_edge_count,
        dtype=np.int64,
    )

    incidence_values = np.ones(
        context_edge_count,
        dtype=np.float32,
    )

    print("Aggregating outgoing edge features...")

    outgoing_incidence = sparse.csr_matrix(
        (
            incidence_values,
            (
                context_sources,
                context_column_ids,
            ),
        ),
        shape=(
            NUM_NODES,
            context_edge_count,
        ),
        dtype=np.float32,
    )

    outgoing_sum = (
        outgoing_incidence @ X_context
    ).toarray().astype(np.float32)

    outgoing_count = np.bincount(
        context_sources,
        minlength=NUM_NODES,
    ).astype(np.float32)

    outgoing_nonzero = outgoing_count > 0

    outgoing_sum[outgoing_nonzero] /= (
        outgoing_count[outgoing_nonzero, None]
    )

    del outgoing_incidence
    gc.collect()

    print("Aggregating incoming edge features...")

    incoming_incidence = sparse.csr_matrix(
        (
            incidence_values,
            (
                context_destinations,
                context_column_ids,
            ),
        ),
        shape=(
            NUM_NODES,
            context_edge_count,
        ),
        dtype=np.float32,
    )

    incoming_sum = (
        incoming_incidence @ X_context
    ).toarray().astype(np.float32)

    incoming_count = np.bincount(
        context_destinations,
        minlength=NUM_NODES,
    ).astype(np.float32)

    incoming_nonzero = incoming_count > 0

    incoming_sum[incoming_nonzero] /= (
        incoming_count[incoming_nonzero, None]
    )

    node_features = np.concatenate(
        [
            outgoing_sum,
            incoming_sum,
        ],
        axis=1,
    ).astype(np.float32)

    edge_index = np.vstack(
        [
            context_sources,
            context_destinations,
        ]
    ).astype(np.int64)

    if not np.isfinite(node_features).all():
        raise ValueError(
            "Non-finite graph node features found."
        )

    graph_checkpoint = {
        "node_features": torch.from_numpy(
            node_features
        ),
        "edge_index": torch.from_numpy(
            edge_index
        ),
    }

    torch.save(
        graph_checkpoint,
        cache_path,
    )

    graph_construction_seconds = (
        time.perf_counter() - build_start
    )

    summary = {
        "seed": int(seed),
        "nodes": NUM_NODES,
        "training_partition_edges": int(
            len(train_transaction_ids)
        ),
        "supervision_edges_removed": int(
            len(supervision_ids)
        ),
        "message_passing_edges": (
            context_edge_count
        ),
        "validation_edges_in_graph": 0,
        "test_edges_in_graph": 0,
        "node_feature_count": (
            NODE_FEATURE_COUNT
        ),
        "directed_graph": True,
        "graph_construction_seconds": float(
            graph_construction_seconds
        ),
    }

    with open(
        summary_path,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            summary,
            file,
            indent=2,
        )

    print(
        "Message-passing edges:",
        f"{context_edge_count:,}",
    )

    print(
        "Supervision edges removed:",
        f"{len(supervision_ids):,}",
    )

    print(
        "Graph construction seconds:",
        f"{graph_construction_seconds:.2f}",
    )

    del X_train_full
    del X_context
    del context_mask
    del context_positions
    del context_sources
    del context_destinations
    del context_column_ids
    del incidence_values
    del outgoing_sum
    del incoming_sum
    del node_features
    del edge_index
    del graph_checkpoint
    gc.collect()

    return summary


for graph_seed in SEEDS:
    build_graph_cache(graph_seed)


# ============================================================
# Model definitions
# ============================================================

class TransactionEdgeGNN(nn.Module):
    def __init__(
        self,
        architecture,
        node_input_dim,
        edge_input_dim,
        hidden_dim,
        num_layers,
        dropout,
        graph_direction,
    ):
        super().__init__()

        self.architecture = architecture
        self.dropout = dropout
        self.graph_direction = graph_direction
        self.convolutions = nn.ModuleList()

        current_dim = node_input_dim

        for _ in range(num_layers):
            if architecture == "GCN":
                convolution = GCNConv(
                    current_dim,
                    hidden_dim,
                    add_self_loops=True,
                    normalize=True,
                )

            elif architecture == "GraphSAGE":
                convolution = SAGEConv(
                    current_dim,
                    hidden_dim,
                    aggr="mean",
                )

            elif architecture == "GAT":
                convolution = GATConv(
                    current_dim,
                    hidden_dim,
                    heads=2,
                    concat=False,
                    dropout=dropout,
                    add_self_loops=True,
                )

            else:
                raise ValueError(
                    "Unknown GNN architecture: "
                    + architecture
                )

            self.convolutions.append(
                convolution
            )

            current_dim = hidden_dim

        decoder_input_dim = (
            2 * hidden_dim
            + edge_input_dim
        )

        self.decoder = nn.Sequential(
            nn.Linear(
                decoder_input_dim,
                hidden_dim,
            ),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(
                hidden_dim,
                1,
            ),
        )

    def prepare_edge_index(
        self,
        edge_index,
    ):
        if self.graph_direction == "directed":
            return edge_index

        if self.graph_direction == "symmetrized":
            reversed_edges = torch.stack(
                [
                    edge_index[1],
                    edge_index[0],
                ],
                dim=0,
            )

            return torch.cat(
                [
                    edge_index,
                    reversed_edges,
                ],
                dim=1,
            )

        raise ValueError(
            "Unknown graph direction."
        )

    def encode(
        self,
        node_features,
        edge_index,
    ):
        message_edges = self.prepare_edge_index(
            edge_index
        )

        hidden = node_features

        for convolution in self.convolutions:
            hidden = convolution(
                hidden,
                message_edges,
            )

            hidden = F.relu(hidden)

            hidden = F.dropout(
                hidden,
                p=self.dropout,
                training=self.training,
            )

        return hidden

    def decode(
        self,
        node_embeddings,
        target_edge_index,
        target_edge_features,
    ):
        source_embeddings = node_embeddings[
            target_edge_index[0]
        ]

        destination_embeddings = node_embeddings[
            target_edge_index[1]
        ]

        decoder_input = torch.cat(
            [
                source_embeddings,
                destination_embeddings,
                target_edge_features,
            ],
            dim=1,
        )

        return self.decoder(
            decoder_input
        ).squeeze(-1)


class EdgeOnlyMLP(nn.Module):
    def __init__(
        self,
        edge_input_dim,
        hidden_dim,
        dropout,
    ):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(
                edge_input_dim,
                hidden_dim,
            ),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(
                hidden_dim,
                hidden_dim,
            ),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(
                hidden_dim,
                1,
            ),
        )

    def forward(self, edge_features):
        return self.network(
            edge_features
        ).squeeze(-1)


# ============================================================
# Reproducibility and graph loading
# ============================================================

def set_gnn_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def load_seed_graph(seed):
    cache_path = graph_cache_path(seed)

    try:
        checkpoint = torch.load(
            cache_path,
            map_location="cpu",
            weights_only=True,
        )
    except TypeError:
        checkpoint = torch.load(
            cache_path,
            map_location="cpu",
        )

    return checkpoint


# ============================================================
# Batched edge scoring
# ============================================================

@torch.no_grad()
def predict_gnn_scores(
    model,
    node_features,
    graph_edge_index,
    target_sources,
    target_destinations,
    target_features,
):
    model.eval()

    torch.cuda.synchronize()
    encoding_start = time.perf_counter()

    node_embeddings = model.encode(
        node_features,
        graph_edge_index,
    )

    torch.cuda.synchronize()

    encoding_seconds = (
        time.perf_counter()
        - encoding_start
    )

    score_parts = []
    first_decode_seconds = None
    total_decode_seconds = 0.0

    for start in range(
        0,
        len(target_sources),
        GNN_SCORE_BATCH_SIZE,
    ):
        end = min(
            start + GNN_SCORE_BATCH_SIZE,
            len(target_sources),
        )

        edge_index_batch = torch.tensor(
            np.vstack(
                [
                    target_sources[start:end],
                    target_destinations[start:end],
                ]
            ),
            dtype=torch.long,
            device=DEVICE,
        )

        feature_batch = torch.tensor(
            np.array(
                target_features[start:end],
                dtype=np.float32,
                copy=True,
            ),
            dtype=torch.float32,
            device=DEVICE,
        )

        torch.cuda.synchronize()
        decode_start = time.perf_counter()

        logits = model.decode(
            node_embeddings,
            edge_index_batch,
            feature_batch,
        )

        probabilities = torch.sigmoid(
            logits
        )

        torch.cuda.synchronize()

        decode_seconds = (
            time.perf_counter()
            - decode_start
        )

        if first_decode_seconds is None:
            first_decode_seconds = (
                decode_seconds
            )

        total_decode_seconds += (
            decode_seconds
        )

        score_parts.append(
            probabilities.detach()
            .cpu()
            .numpy()
        )

        del edge_index_batch
        del feature_batch
        del logits
        del probabilities

    scores = np.concatenate(
        score_parts
    ).astype(np.float32)

    del node_embeddings
    del score_parts
    torch.cuda.empty_cache()

    return {
        "scores": scores,
        "encoding_seconds": (
            encoding_seconds
        ),
        "first_decode_seconds": float(
            first_decode_seconds or 0.0
        ),
        "total_decode_seconds": (
            total_decode_seconds
        ),
    }


@torch.no_grad()
def predict_edge_only_scores(
    model,
    target_features,
):
    model.eval()

    score_parts = []
    first_batch_seconds = None
    total_seconds = 0.0

    for start in range(
        0,
        len(target_features),
        GNN_SCORE_BATCH_SIZE,
    ):
        end = min(
            start + GNN_SCORE_BATCH_SIZE,
            len(target_features),
        )

        feature_batch = torch.tensor(
            np.array(
                target_features[start:end],
                dtype=np.float32,
                copy=True,
            ),
            dtype=torch.float32,
            device=DEVICE,
        )

        torch.cuda.synchronize()
        batch_start = time.perf_counter()

        probabilities = torch.sigmoid(
            model(feature_batch)
        )

        torch.cuda.synchronize()

        batch_seconds = (
            time.perf_counter()
            - batch_start
        )

        if first_batch_seconds is None:
            first_batch_seconds = (
                batch_seconds
            )

        total_seconds += batch_seconds

        score_parts.append(
            probabilities.detach()
            .cpu()
            .numpy()
        )

        del feature_batch
        del probabilities

    return {
        "scores": np.concatenate(
            score_parts
        ).astype(np.float32),
        "encoding_seconds": 0.0,
        "first_decode_seconds": float(
            first_batch_seconds or 0.0
        ),
        "total_decode_seconds": (
            total_seconds
        ),
    }


# ============================================================
# One complete GNN training run
# ============================================================

def train_gnn_once(
    architecture,
    configuration,
    seed,
    maximum_epochs,
    patience,
    history_path,
):
    set_gnn_seed(seed)

    graph = load_seed_graph(seed)

    node_features = graph[
        "node_features"
    ].to(
        DEVICE,
        non_blocking=True,
    )

    graph_edge_index = graph[
        "edge_index"
    ].to(
        DEVICE,
        non_blocking=True,
    )

    training_ids = np.load(
        matrix_directory
        / f"train_seed_{seed}_transaction_ids.npy"
    ).astype(np.int64)

    training_labels = np.load(
        matrix_directory
        / f"train_seed_{seed}_labels.npy"
    ).astype(np.float32)

    training_sources = df.iloc[
        training_ids
    ]["Source Node ID"].to_numpy(
        dtype=np.int64
    )

    training_destinations = df.iloc[
        training_ids
    ]["Destination Node ID"].to_numpy(
        dtype=np.int64
    )

    X_training_sparse = sparse.load_npz(
        matrix_directory
        / f"train_seed_{seed}.npz"
    )

    training_edge_features = (
        X_training_sparse
        .toarray()
        .astype(np.float32)
    )

    training_edge_index = torch.tensor(
        np.vstack(
            [
                training_sources,
                training_destinations,
            ]
        ),
        dtype=torch.long,
        device=DEVICE,
    )

    training_feature_tensor = torch.tensor(
        training_edge_features,
        dtype=torch.float32,
        device=DEVICE,
    )

    training_label_tensor = torch.tensor(
        training_labels,
        dtype=torch.float32,
        device=DEVICE,
    )

    model = TransactionEdgeGNN(
        architecture=architecture,
        node_input_dim=NODE_FEATURE_COUNT,
        edge_input_dim=EDGE_FEATURE_COUNT,
        hidden_dim=configuration[
            "hidden_dim"
        ],
        num_layers=configuration[
            "num_layers"
        ],
        dropout=configuration["dropout"],
        graph_direction=configuration[
            "graph_direction"
        ],
    ).to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=configuration["learning_rate"],
        weight_decay=configuration[
            "weight_decay"
        ],
    )

    positive_count = float(
        training_labels.sum()
    )

    negative_count = float(
        len(training_labels)
        - positive_count
    )

    positive_weight = torch.tensor(
        [negative_count / positive_count],
        dtype=torch.float32,
        device=DEVICE,
    )

    loss_function = nn.BCEWithLogitsLoss(
        pos_weight=positive_weight
    )

    best_validation_auprc = -np.inf
    best_epoch = 0
    best_state = None
    best_validation_scores = None
    epochs_without_improvement = 0
    history_records = []
    total_training_seconds = 0.0
    total_validation_seconds = 0.0

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    for epoch in range(
        1,
        maximum_epochs + 1,
    ):
        model.train()
        optimizer.zero_grad(set_to_none=True)

        torch.cuda.synchronize()
        epoch_start = time.perf_counter()

        node_embeddings = model.encode(
            node_features,
            graph_edge_index,
        )

        training_logits = model.decode(
            node_embeddings,
            training_edge_index,
            training_feature_tensor,
        )

        loss = loss_function(
            training_logits,
            training_label_tensor,
        )

        loss.backward()
        optimizer.step()

        torch.cuda.synchronize()

        epoch_training_seconds = (
            time.perf_counter()
            - epoch_start
        )

        total_training_seconds += (
            epoch_training_seconds
        )

        validation_start = time.perf_counter()

        validation_output = predict_gnn_scores(
            model,
            node_features,
            graph_edge_index,
            validation_sources,
            validation_destinations,
            validation_dense,
        )

        epoch_validation_seconds = (
            time.perf_counter()
            - validation_start
        )

        total_validation_seconds += (
            epoch_validation_seconds
        )

        validation_auprc = (
            average_precision_score(
                y_validation,
                validation_output["scores"],
            )
        )

        validation_roc_auc = roc_auc_score(
            y_validation,
            validation_output["scores"],
        )

        history_records.append({
            "epoch": epoch,
            "training_loss": float(
                loss.item()
            ),
            "validation_auprc": float(
                validation_auprc
            ),
            "validation_roc_auc": float(
                validation_roc_auc
            ),
            "training_seconds": float(
                epoch_training_seconds
            ),
            "validation_seconds": float(
                epoch_validation_seconds
            ),
        })

        pd.DataFrame(
            history_records
        ).to_csv(
            history_path,
            index=False,
        )

        print(
            f"Epoch {epoch}: "
            f"loss={loss.item():.6f}, "
            f"validation AUPRC="
            f"{validation_auprc:.6f}"
        )

        if (
            validation_auprc
            > best_validation_auprc
            + 1e-8
        ):
            best_validation_auprc = float(
                validation_auprc
            )

            best_epoch = int(epoch)

            best_state = {
                key: value.detach()
                .cpu()
                .clone()
                for key, value
                in model.state_dict().items()
            }

            best_validation_scores = (
                validation_output["scores"]
                .copy()
            )

            epochs_without_improvement = 0

        else:
            epochs_without_improvement += 1

        del node_embeddings
        del training_logits
        del loss
        del validation_output
        torch.cuda.empty_cache()

        if (
            epochs_without_improvement
            >= patience
        ):
            print(
                "Validation early stopping "
                f"at epoch {epoch}."
            )
            break

    if best_state is None:
        raise RuntimeError(
            "No valid GNN checkpoint was produced."
        )

    model.load_state_dict(best_state)

    peak_gpu_memory_gb = (
        torch.cuda.max_memory_allocated()
        / (1024 ** 3)
    )

    return {
        "model": model,
        "node_features": node_features,
        "graph_edge_index": (
            graph_edge_index
        ),
        "best_epoch": best_epoch,
        "best_validation_auprc": (
            best_validation_auprc
        ),
        "best_validation_scores": (
            best_validation_scores
        ),
        "training_seconds": (
            total_training_seconds
        ),
        "validation_seconds": (
            total_validation_seconds
        ),
        "peak_gpu_memory_gb": (
            peak_gpu_memory_gb
        ),
        "training_transaction_count": int(
            len(training_labels)
        ),
        "training_positive_count": int(
            positive_count
        ),
    }


def release_gnn_run(run):
    run["model"].cpu()

    del run["model"]
    del run["node_features"]
    del run["graph_edge_index"]
    del run["best_validation_scores"]

    torch.cuda.empty_cache()
    gc.collect()


# ============================================================
# Minimal GNN hyperparameter search
# ============================================================

GNN_CANDIDATES = [
    {
        "hidden_dim": 32,
        "num_layers": 2,
        "dropout": 0.20,
        "learning_rate": 0.001,
        "weight_decay": 0.0001,
        "graph_direction": "directed",
    },
    {
        "hidden_dim": 64,
        "num_layers": 2,
        "dropout": 0.30,
        "learning_rate": 0.0005,
        "weight_decay": 0.0001,
        "graph_direction": "directed",
    },
]

architectures = [
    "GCN",
    "GraphSAGE",
    "GAT",
]

selected_gnn_configurations = {}

for architecture in architectures:
    architecture_key = (
        architecture.lower()
        .replace("graphsage", "graphsage")
    )

    architecture_directory = (
        gnn_root / architecture_key
    )

    architecture_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    selection_path = (
        architecture_directory
        / "selected_hyperparameters.json"
    )

    search_result_path = (
        architecture_directory
        / "hyperparameter_search.csv"
    )

    if (
        selection_path.exists()
        and search_result_path.exists()
    ):
        print(
            f"\nCompleted search found for "
            f"{architecture}."
        )

        with open(
            selection_path,
            "r",
            encoding="utf-8",
        ) as file:
            saved_selection = json.load(file)

        selected_gnn_configurations[
            architecture
        ] = saved_selection[
            "best_configuration"
        ]

        display(
            pd.read_csv(search_result_path)
        )

        continue

    search_records = []
    search_start = time.perf_counter()

    for candidate_number, configuration in enumerate(
        GNN_CANDIDATES,
        start=1,
    ):
        print(
            f"\n{architecture} candidate "
            f"{candidate_number}"
        )

        history_path = (
            architecture_directory
            / f"search_candidate_"
            f"{candidate_number}_history.csv"
        )

        with PeakMemoryMonitor() as memory_monitor:
            run = train_gnn_once(
                architecture=architecture,
                configuration=configuration,
                seed=42,
                maximum_epochs=(
                    GNN_SEARCH_EPOCHS
                ),
                patience=GNN_PATIENCE,
                history_path=history_path,
            )

        search_records.append({
            "candidate": candidate_number,
            "configuration_json": json.dumps(
                configuration,
                sort_keys=True,
            ),
            "best_epoch": run["best_epoch"],
            "validation_auprc": (
                run["best_validation_auprc"]
            ),
            "training_seconds": (
                run["training_seconds"]
            ),
            "validation_seconds": (
                run["validation_seconds"]
            ),
            "peak_cpu_memory_gb": (
                memory_monitor.peak_gb
            ),
            "peak_gpu_memory_gb": (
                run["peak_gpu_memory_gb"]
            ),
        })

        pd.DataFrame(
            search_records
        ).to_csv(
            search_result_path,
            index=False,
        )

        release_gnn_run(run)

    search_results = pd.DataFrame(
        search_records
    )

    best_search_row = (
        search_results
        .sort_values(
            "validation_auprc",
            ascending=False,
        )
        .iloc[0]
    )

    best_candidate = int(
        best_search_row["candidate"]
    )

    best_configuration = copy.deepcopy(
        GNN_CANDIDATES[
            best_candidate - 1
        ]
    )

    selection = {
        "architecture": architecture,
        "selection_metric": (
            "validation_auprc"
        ),
        "search_seed": 42,
        "search_epochs_per_candidate": (
            GNN_SEARCH_EPOCHS
        ),
        "search_seconds": float(
            time.perf_counter()
            - search_start
        ),
        "candidates": GNN_CANDIDATES,
        "best_candidate": best_candidate,
        "best_configuration": (
            best_configuration
        ),
        "best_validation_auprc": float(
            best_search_row[
                "validation_auprc"
            ]
        ),
    }

    with open(
        selection_path,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            selection,
            file,
            indent=2,
        )

    selected_gnn_configurations[
        architecture
    ] = best_configuration

    print(
        f"\n{architecture} search results"
    )

    display(search_results)

    print(
        "Selected configuration:",
        best_configuration,
    )


# ============================================================
# Final five-seed GNN runs
# ============================================================

all_gnn_result_tables = []

for architecture in architectures:
    architecture_key = architecture.lower()

    architecture_directory = (
        gnn_root / architecture_key
    )

    seed_result_directory = (
        architecture_directory
        / "seed_results"
    )

    seed_result_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    model_directory = (
        architecture_directory / "checkpoints"
    )

    model_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    metrics_path = (
        architecture_directory
        / "metrics_by_seed.csv"
    )

    result_records = []

    selected_configuration = (
        selected_gnn_configurations[
            architecture
        ]
    )

    for seed in SEEDS:
        print(
            f"\nFinal {architecture} seed {seed}"
        )

        model_path = (
            model_directory
            / f"{architecture_key}_seed_"
            f"{seed}.pt"
        )

        prediction_path = (
            gnn_prediction_directory
            / f"{architecture_key}_test_"
            f"predictions_seed_{seed}.parquet"
        )

        seed_result_path = (
            seed_result_directory
            / f"seed_{seed}.json"
        )

        if (
            model_path.exists()
            and prediction_path.exists()
            and seed_result_path.exists()
        ):
            print(
                "Completed checkpoint found. "
                "Skipping this seed."
            )

            with open(
                seed_result_path,
                "r",
                encoding="utf-8",
            ) as file:
                record = json.load(file)

            result_records.append(record)
            continue

        history_path = (
            seed_result_directory
            / f"training_history_seed_"
            f"{seed}.csv"
        )

        with PeakMemoryMonitor() as memory_monitor:
            run = train_gnn_once(
                architecture=architecture,
                configuration=(
                    selected_configuration
                ),
                seed=seed,
                maximum_epochs=(
                    GNN_FINAL_EPOCHS
                ),
                patience=GNN_PATIENCE,
                history_path=history_path,
            )

            threshold_result = (
                select_validation_threshold(
                    y_validation,
                    run[
                        "best_validation_scores"
                    ],
                )
            )

            validation_metrics = (
                calculate_metrics(
                    y_validation,
                    run[
                        "best_validation_scores"
                    ],
                    threshold_result[
                        "threshold"
                    ],
                )
            )

            test_output = predict_gnn_scores(
                run["model"],
                run["node_features"],
                run["graph_edge_index"],
                test_sources,
                test_destinations,
                test_dense,
            )

        selected_threshold = (
            threshold_result["threshold"]
        )

        test_scores = test_output["scores"]

        test_metrics = calculate_metrics(
            y_test,
            test_scores,
            selected_threshold,
        )

        dense_metrics = calculate_metrics(
            y_test[dense_test_mask],
            test_scores[dense_test_mask],
            selected_threshold,
        )

        tail_metrics = calculate_metrics(
            y_test[tail_test_mask],
            test_scores[tail_test_mask],
            selected_threshold,
        )

        predicted_labels = (
            test_scores
            >= selected_threshold
        ).astype(np.int8)

        prediction_table = pd.DataFrame({
            "Transaction ID": (
                test_transaction_ids
            ),
            "true_label": y_test,
            "prediction_score": (
                test_scores
            ),
            "predicted_label": (
                predicted_labels
            ),
            "test_period": np.where(
                dense_test_mask,
                "dense_period",
                "timestamp_tail",
            ),
        })

        prediction_table.to_parquet(
            prediction_path,
            index=False,
        )

        model_checkpoint = {
            "architecture": architecture,
            "configuration": (
                selected_configuration
            ),
            "node_input_dim": (
                NODE_FEATURE_COUNT
            ),
            "edge_input_dim": (
                EDGE_FEATURE_COUNT
            ),
            "seed": int(seed),
            "best_epoch": int(
                run["best_epoch"]
            ),
            "state_dict": {
                key: value.detach().cpu()
                for key, value
                in run["model"]
                .state_dict()
                .items()
            },
        }

        torch.save(
            model_checkpoint,
            model_path,
        )

        graph_summary = build_graph_cache(seed)

        record = {
            "model": architecture,
            "seed": int(seed),
            "best_epoch": int(
                run["best_epoch"]
            ),
            "training_transactions": (
                run[
                    "training_transaction_count"
                ]
            ),
            "training_positives": (
                run[
                    "training_positive_count"
                ]
            ),
            "training_seconds": float(
                run["training_seconds"]
            ),
            "validation_inference_seconds": float(
                run["validation_seconds"]
            ),
            "graph_construction_seconds": float(
                graph_summary[
                    "graph_construction_seconds"
                ]
            ),
            "fixed_inference_transactions": int(
                min(
                    GNN_SCORE_BATCH_SIZE,
                    len(y_test),
                )
            ),
            "graph_embedding_seconds": float(
                test_output[
                    "encoding_seconds"
                ]
            ),
            "fixed_inference_seconds": float(
                test_output[
                    "encoding_seconds"
                ]
                + test_output[
                    "first_decode_seconds"
                ]
            ),
            "full_test_inference_seconds": float(
                test_output[
                    "encoding_seconds"
                ]
                + test_output[
                    "total_decode_seconds"
                ]
            ),
            "peak_cpu_memory_gb": float(
                memory_monitor.peak_gb
            ),
            "peak_gpu_memory_gb": float(
                run["peak_gpu_memory_gb"]
            ),
            "validation_selected_threshold": float(
                selected_threshold
            ),
        }

        add_metric_prefix(
            record,
            "validation",
            validation_metrics,
        )

        add_metric_prefix(
            record,
            "test",
            test_metrics,
        )

        add_metric_prefix(
            record,
            "dense_test",
            dense_metrics,
        )

        add_metric_prefix(
            record,
            "tail_test",
            tail_metrics,
        )

        with open(
            seed_result_path,
            "w",
            encoding="utf-8",
        ) as file:
            json.dump(
                record,
                file,
                indent=2,
            )

        result_records.append(record)

        pd.DataFrame(
            result_records
        ).sort_values(
            "seed"
        ).to_csv(
            metrics_path,
            index=False,
        )

        print(
            "Best epoch:",
            run["best_epoch"],
        )

        print(
            "Selected threshold:",
            f"{selected_threshold:.6f}",
        )

        print(
            "Test precision:",
            f"{test_metrics['precision']:.6f}",
        )

        print(
            "Test recall:",
            f"{test_metrics['recall']:.6f}",
        )

        print(
            "Test F1:",
            f"{test_metrics['f1']:.6f}",
        )

        print(
            "Test AUPRC:",
            f"{test_metrics['auprc']:.6f}",
        )

        print(
            "Test ROC-AUC:",
            f"{test_metrics['roc_auc']:.6f}",
        )

        print("Checkpoint saved.")

        del test_output
        del test_scores
        del predicted_labels
        del prediction_table
        del model_checkpoint

        release_gnn_run(run)

    architecture_results = (
        pd.DataFrame(result_records)
        .sort_values("seed")
        .reset_index(drop=True)
    )

    architecture_results.to_csv(
        metrics_path,
        index=False,
    )

    all_gnn_result_tables.append(
        architecture_results
    )

    print(
        f"\n{architecture.upper()} RESULTS"
    )

    display(
        architecture_results[
            [
                "seed",
                "best_epoch",
                "validation_selected_threshold",
                "test_precision",
                "test_recall",
                "test_f1",
                "test_auprc",
                "test_roc_auc",
                "test_true_negatives",
                "test_false_positives",
                "test_false_negatives",
                "test_true_positives",
                "training_seconds",
                "fixed_inference_seconds",
                "peak_cpu_memory_gb",
                "peak_gpu_memory_gb",
            ]
        ]
    )


gnn_results = pd.concat(
    all_gnn_result_tables,
    ignore_index=True,
)

gnn_results.to_csv(
    OUTPUT_DIR / "gnn_metrics_by_seed.csv",
    index=False,
)


# ============================================================
# Select best validation-ranked GNN family
# ============================================================

seed_42_results = gnn_results[
    gnn_results["seed"].eq(42)
]

best_gnn_row = (
    seed_42_results
    .sort_values(
        "validation_auprc",
        ascending=False,
    )
    .iloc[0]
)

best_gnn_architecture = str(
    best_gnn_row["model"]
)

best_gnn_configuration = copy.deepcopy(
    selected_gnn_configurations[
        best_gnn_architecture
    ]
)

print(
    "\nBest validation-ranked GNN:",
    best_gnn_architecture,
)

print(
    "Selected configuration:",
    best_gnn_configuration,
)


# ============================================================
# Edge-only ablation
# ============================================================

def run_edge_only_ablation():
    set_gnn_seed(42)

    X_training = sparse.load_npz(
        matrix_directory
        / "train_seed_42.npz"
    ).toarray().astype(np.float32)

    y_training = np.load(
        matrix_directory
        / "train_seed_42_labels.npy"
    ).astype(np.float32)

    model = EdgeOnlyMLP(
        edge_input_dim=EDGE_FEATURE_COUNT,
        hidden_dim=best_gnn_configuration[
            "hidden_dim"
        ],
        dropout=best_gnn_configuration[
            "dropout"
        ],
    ).to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=best_gnn_configuration[
            "learning_rate"
        ],
        weight_decay=best_gnn_configuration[
            "weight_decay"
        ],
    )

    positive_count = float(
        y_training.sum()
    )

    negative_count = float(
        len(y_training) - positive_count
    )

    loss_function = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(
            [
                negative_count
                / positive_count
            ],
            device=DEVICE,
        )
    )

    training_tensor = torch.tensor(
        X_training,
        dtype=torch.float32,
        device=DEVICE,
    )

    label_tensor = torch.tensor(
        y_training,
        dtype=torch.float32,
        device=DEVICE,
    )

    best_state = None
    best_validation_scores = None
    best_validation_auprc = -np.inf
    best_epoch = 0
    total_training_seconds = 0.0

    for epoch in range(
        1,
        GNN_FINAL_EPOCHS + 1,
    ):
        model.train()
        optimizer.zero_grad(set_to_none=True)

        torch.cuda.synchronize()
        training_start = time.perf_counter()

        logits = model(training_tensor)
        loss = loss_function(
            logits,
            label_tensor,
        )

        loss.backward()
        optimizer.step()

        torch.cuda.synchronize()

        total_training_seconds += (
            time.perf_counter()
            - training_start
        )

        validation_output = (
            predict_edge_only_scores(
                model,
                validation_dense,
            )
        )

        validation_auprc = (
            average_precision_score(
                y_validation,
                validation_output[
                    "scores"
                ],
            )
        )

        if (
            validation_auprc
            > best_validation_auprc
        ):
            best_validation_auprc = float(
                validation_auprc
            )

            best_epoch = epoch

            best_validation_scores = (
                validation_output["scores"]
                .copy()
            )

            best_state = {
                key: value.detach()
                .cpu()
                .clone()
                for key, value
                in model.state_dict().items()
            }

    model.load_state_dict(best_state)

    threshold_result = (
        select_validation_threshold(
            y_validation,
            best_validation_scores,
        )
    )

    test_output = predict_edge_only_scores(
        model,
        test_dense,
    )

    test_metrics = calculate_metrics(
        y_test,
        test_output["scores"],
        threshold_result["threshold"],
    )

    torch.save(
        {
            "state_dict": best_state,
            "best_epoch": best_epoch,
            "configuration": (
                best_gnn_configuration
            ),
        },
        gnn_root
        / "ablation_edge_only_seed_42.pt",
    )

    model.cpu()
    torch.cuda.empty_cache()

    return {
        "ablation": (
            "edge_attributes_without_"
            "message_passing"
        ),
        "architecture": "Edge-only MLP",
        "seed": 42,
        "num_layers": 0,
        "graph_direction": "none",
        "best_epoch": best_epoch,
        "validation_auprc": (
            best_validation_auprc
        ),
        "test_precision": (
            test_metrics["precision"]
        ),
        "test_recall": (
            test_metrics["recall"]
        ),
        "test_f1": test_metrics["f1"],
        "test_auprc": (
            test_metrics["auprc"]
        ),
        "test_roc_auc": (
            test_metrics["roc_auc"]
        ),
        "training_seconds": (
            total_training_seconds
        ),
    }


# ============================================================
# Direction and depth ablations
# ============================================================

ablation_directory = (
    gnn_root / "ablations"
)

ablation_directory.mkdir(
    parents=True,
    exist_ok=True,
)

ablation_result_path = (
    ablation_directory
    / "gnn_ablation_results.csv"
)

ablation_records = []

existing_main_row = (
    gnn_results[
        gnn_results["model"].eq(
            best_gnn_architecture
        )
        & gnn_results["seed"].eq(42)
    ]
    .iloc[0]
)

ablation_records.append({
    "ablation": "directed_two_layers",
    "architecture": (
        best_gnn_architecture
    ),
    "seed": 42,
    "num_layers": 2,
    "graph_direction": "directed",
    "best_epoch": int(
        existing_main_row["best_epoch"]
    ),
    "validation_auprc": float(
        existing_main_row[
            "validation_auprc"
        ]
    ),
    "test_precision": float(
        existing_main_row["test_precision"]
    ),
    "test_recall": float(
        existing_main_row["test_recall"]
    ),
    "test_f1": float(
        existing_main_row["test_f1"]
    ),
    "test_auprc": float(
        existing_main_row["test_auprc"]
    ),
    "test_roc_auc": float(
        existing_main_row["test_roc_auc"]
    ),
    "training_seconds": float(
        existing_main_row[
            "training_seconds"
        ]
    ),
})

edge_only_result = run_edge_only_ablation()
ablation_records.append(
    edge_only_result
)

ablation_variants = [
    {
        "ablation": "directed_one_layer",
        "num_layers": 1,
        "graph_direction": "directed",
    },
    {
        "ablation": (
            "symmetrized_two_layers"
        ),
        "num_layers": 2,
        "graph_direction": "symmetrized",
    },
    {
        "ablation": "directed_three_layers",
        "num_layers": 3,
        "graph_direction": "directed",
    },
]

for variant in ablation_variants:
    print(
        "\nRunning ablation:",
        variant["ablation"],
    )

    configuration = copy.deepcopy(
        best_gnn_configuration
    )

    configuration["num_layers"] = (
        variant["num_layers"]
    )

    configuration["graph_direction"] = (
        variant["graph_direction"]
    )

    history_path = (
        ablation_directory
        / f"{variant['ablation']}_history.csv"
    )

    run = train_gnn_once(
        architecture=best_gnn_architecture,
        configuration=configuration,
        seed=42,
        maximum_epochs=GNN_FINAL_EPOCHS,
        patience=GNN_PATIENCE,
        history_path=history_path,
    )

    threshold_result = (
        select_validation_threshold(
            y_validation,
            run["best_validation_scores"],
        )
    )

    test_output = predict_gnn_scores(
        run["model"],
        run["node_features"],
        run["graph_edge_index"],
        test_sources,
        test_destinations,
        test_dense,
    )

    test_metrics = calculate_metrics(
        y_test,
        test_output["scores"],
        threshold_result["threshold"],
    )

    ablation_records.append({
        "ablation": variant["ablation"],
        "architecture": (
            best_gnn_architecture
        ),
        "seed": 42,
        "num_layers": (
            variant["num_layers"]
        ),
        "graph_direction": (
            variant["graph_direction"]
        ),
        "best_epoch": run["best_epoch"],
        "validation_auprc": (
            run["best_validation_auprc"]
        ),
        "test_precision": (
            test_metrics["precision"]
        ),
        "test_recall": (
            test_metrics["recall"]
        ),
        "test_f1": test_metrics["f1"],
        "test_auprc": (
            test_metrics["auprc"]
        ),
        "test_roc_auc": (
            test_metrics["roc_auc"]
        ),
        "training_seconds": (
            run["training_seconds"]
        ),
    })

    checkpoint_path = (
        ablation_directory
        / f"{variant['ablation']}.pt"
    )

    torch.save(
        {
            "architecture": (
                best_gnn_architecture
            ),
            "configuration": configuration,
            "best_epoch": run["best_epoch"],
            "state_dict": {
                key: value.detach().cpu()
                for key, value
                in run["model"]
                .state_dict()
                .items()
            },
        },
        checkpoint_path,
    )

    pd.DataFrame(
        ablation_records
    ).to_csv(
        ablation_result_path,
        index=False,
    )

    release_gnn_run(run)

abDf = None

ablation_results = pd.DataFrame(
    ablation_records
)

ablation_results.to_csv(
    ablation_result_path,
    index=False,
)


# ============================================================
# Combined output
# ============================================================

combined_model_results = pd.concat(
    [
        conventional_results,
        gnn_results,
    ],
    ignore_index=True,
)

combined_model_results.to_csv(
    OUTPUT_DIR
    / "all_models_metrics_by_seed.csv",
    index=False,
)

print("\nALL GNN RESULTS")

display(
    gnn_results[
        [
            "model",
            "seed",
            "best_epoch",
            "validation_selected_threshold",
            "test_precision",
            "test_recall",
            "test_f1",
            "test_auprc",
            "test_roc_auc",
            "test_true_negatives",
            "test_false_positives",
            "test_false_negatives",
            "test_true_positives",
            "training_seconds",
            "fixed_inference_seconds",
            "peak_cpu_memory_gb",
            "peak_gpu_memory_gb",
        ]
    ]
)

print("\nGNN ABLATION RESULTS")
display(ablation_results)

print(
    "\nSaved all GNN results to:",
    OUTPUT_DIR / "gnn_metrics_by_seed.csv",
)

print(
    "Saved combined results to:",
    OUTPUT_DIR
    / "all_models_metrics_by_seed.csv",
)

## Converged GNN runs

Extends GNN training with validation early stopping and saves final checkpoints.


In [ ]:
from pathlib import Path
import copy
import gc
import json

import numpy as np
import pandas as pd
import torch

CONVERGENCE_MAX_EPOCHS = 40
CONVERGENCE_PATIENCE = 5

converged_root = (
    OUTPUT_DIR / "gnn_converged"
)

converged_root.mkdir(
    parents=True,
    exist_ok=True,
)

converged_records = []

for architecture in architectures:
    architecture_key = architecture.lower()

    architecture_directory = (
        converged_root / architecture_key
    )

    architecture_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    checkpoint_directory = (
        architecture_directory / "checkpoints"
    )

    checkpoint_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    validation_score_directory = (
        architecture_directory
        / "validation_scores"
    )

    validation_score_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    seed_result_directory = (
        architecture_directory
        / "seed_results"
    )

    seed_result_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    selected_configuration = copy.deepcopy(
        selected_gnn_configurations[
            architecture
        ]
    )

    for seed in SEEDS:
        print(
            f"\nConverged {architecture} "
            f"seed {seed}"
        )

        checkpoint_path = (
            checkpoint_directory
            / f"{architecture_key}_seed_"
            f"{seed}.pt"
        )

        validation_score_path = (
            validation_score_directory
            / f"validation_scores_seed_"
            f"{seed}.npy"
        )

        result_path = (
            seed_result_directory
            / f"seed_{seed}.json"
        )

        history_path = (
            seed_result_directory
            / f"training_history_seed_"
            f"{seed}.csv"
        )

        if (
            checkpoint_path.exists()
            and validation_score_path.exists()
            and result_path.exists()
        ):
            print(
                "Completed convergence checkpoint "
                "found. Skipping."
            )

            with open(
                result_path,
                "r",
                encoding="utf-8",
            ) as file:
                converged_records.append(
                    json.load(file)
                )

            continue

        with PeakMemoryMonitor() as memory_monitor:
            run = train_gnn_once(
                architecture=architecture,
                configuration=(
                    selected_configuration
                ),
                seed=seed,
                maximum_epochs=(
                    CONVERGENCE_MAX_EPOCHS
                ),
                patience=(
                    CONVERGENCE_PATIENCE
                ),
                history_path=history_path,
            )

        validation_scores = run[
            "best_validation_scores"
        ]

        threshold_result = (
            select_validation_threshold(
                y_validation,
                validation_scores,
            )
        )

        validation_metrics = calculate_metrics(
            y_validation,
            validation_scores,
            threshold_result["threshold"],
        )

        np.save(
            validation_score_path,
            validation_scores.astype(
                np.float32
            ),
        )

        checkpoint = {
            "architecture": architecture,
            "configuration": (
                selected_configuration
            ),
            "seed": int(seed),
            "best_epoch": int(
                run["best_epoch"]
            ),
            "node_input_dim": (
                NODE_FEATURE_COUNT
            ),
            "edge_input_dim": (
                EDGE_FEATURE_COUNT
            ),
            "state_dict": {
                key: value.detach().cpu()
                for key, value
                in run["model"]
                .state_dict()
                .items()
            },
        }

        torch.save(
            checkpoint,
            checkpoint_path,
        )

        graph_summary = build_graph_cache(seed)

        record = {
            "model": architecture,
            "seed": int(seed),
            "best_epoch": int(
                run["best_epoch"]
            ),
            "maximum_epochs": (
                CONVERGENCE_MAX_EPOCHS
            ),
            "early_stopping_patience": (
                CONVERGENCE_PATIENCE
            ),
            "validation_selected_threshold": float(
                threshold_result["threshold"]
            ),
            "validation_accuracy": (
                validation_metrics["accuracy"]
            ),
            "validation_precision": (
                validation_metrics["precision"]
            ),
            "validation_recall": (
                validation_metrics["recall"]
            ),
            "validation_f1": (
                validation_metrics["f1"]
            ),
            "validation_auprc": (
                validation_metrics["auprc"]
            ),
            "validation_roc_auc": (
                validation_metrics["roc_auc"]
            ),
            "training_seconds": float(
                run["training_seconds"]
            ),
            "validation_inference_seconds": float(
                run["validation_seconds"]
            ),
            "graph_construction_seconds": float(
                graph_summary[
                    "graph_construction_seconds"
                ]
            ),
            "peak_cpu_memory_gb": float(
                memory_monitor.peak_gb
            ),
            "peak_gpu_memory_gb": float(
                run["peak_gpu_memory_gb"]
            ),
            "test_accessed": False,
        }

        with open(
            result_path,
            "w",
            encoding="utf-8",
        ) as file:
            json.dump(
                record,
                file,
                indent=2,
            )

        converged_records.append(record)

        pd.DataFrame(
            converged_records
        ).sort_values(
            ["model", "seed"]
        ).to_csv(
            converged_root
            / "validation_results.csv",
            index=False,
        )

        print(
            "Best epoch:",
            run["best_epoch"],
        )

        print(
            "Validation AUPRC:",
            f"{validation_metrics['auprc']:.6f}",
        )

        print(
            "Validation F1:",
            f"{validation_metrics['f1']:.6f}",
        )

        print(
            "Selected threshold:",
            f"{threshold_result['threshold']:.6f}",
        )

        print(
            "Test accessed: False"
        )

        del checkpoint
        del validation_scores

        release_gnn_run(run)
        gc.collect()

converged_validation_results = (
    pd.DataFrame(converged_records)
    .sort_values(["model", "seed"])
    .reset_index(drop=True)
)

converged_validation_results.to_csv(
    converged_root
    / "validation_results.csv",
    index=False,
)

print(
    "\nCONVERGED GNN VALIDATION RESULTS"
)

display(
    converged_validation_results[
        [
            "model",
            "seed",
            "best_epoch",
            "validation_selected_threshold",
            "validation_precision",
            "validation_recall",
            "validation_f1",
            "validation_auprc",
            "validation_roc_auc",
            "training_seconds",
            "peak_gpu_memory_gb",
            "test_accessed",
        ]
    ]
)

print(
    "\nNo test predictions were generated "
    "by this cell."
)

## Final tables, figures, predictions, and package

Evaluates converged checkpoints and assembles the final reproducibility outputs.


In [ ]:
from pathlib import Path
import gc
import io
import json
import math
import os
import platform
import shutil
import sys
import time
import warnings
import zipfile

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sklearn
import torch
import xgboost as xgb
from scipy import stats
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

warnings.filterwarnings("ignore")

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

ROOT = Path("/kaggle/working/aml_outputs")
FINAL = ROOT / "final_delivery"
FIGURES = FINAL / "figures"
EXPLANATIONS = FINAL / "explainability"
PREDICTIONS = FINAL / "predictions"
TABLES = FINAL / "tables"
for folder in [FINAL, FIGURES, EXPLANATIONS, PREDICTIONS, TABLES]:
    folder.mkdir(parents=True, exist_ok=True)

SEED_LIST = [42, 52, 62, 72, 82]
MODEL_KEYS = {"GCN": "gcn", "GraphSAGE": "graphsage", "GAT": "gat"}


def clean_text(value):
    return str(value).replace(chr(8212), "-").replace("-" * 3, "- - -")


def metric_set(labels, scores, threshold):
    labels = np.asarray(labels, dtype=np.int8)
    scores = np.asarray(scores, dtype=np.float64)
    predictions = (scores >= float(threshold)).astype(np.int8)
    tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()
    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision_score(labels, predictions, zero_division=0),
        "recall": recall_score(labels, predictions, zero_division=0),
        "f1": f1_score(labels, predictions, zero_division=0),
        "auprc": average_precision_score(labels, scores),
        "roc_auc": roc_auc_score(labels, scores),
        "true_negatives": int(tn),
        "false_positives": int(fp),
        "false_negatives": int(fn),
        "true_positives": int(tp),
    }, predictions


def record_value(record, names, default=None):
    for name in names:
        if name in record:
            return record[name]
    return default


def object_value(obj, names, predicate=None):
    if isinstance(obj, dict):
        for name in names:
            if name in obj:
                return obj[name]
        values = list(obj.values())
    else:
        for name in names:
            if hasattr(obj, name):
                return getattr(obj, name)
        values = list(obj) if isinstance(obj, (tuple, list)) else []
    if predicate is not None:
        for value in values:
            try:
                if predicate(value):
                    return value
            except Exception:
                pass
    raise KeyError("Could not locate any of: " + ", ".join(names))


def graph_components(graph_run):
    node_features = object_value(
        graph_run,
        ["node_features", "node_x", "x"],
        lambda value: hasattr(value, "ndim") and value.ndim == 2 and value.shape[1] == 108,
    )
    graph_edge_index = object_value(
        graph_run,
        ["graph_edge_index", "message_edge_index", "edge_index"],
        lambda value: hasattr(value, "ndim") and value.ndim == 2 and value.shape[0] == 2 and value.shape[1] > 1000000,
    )
    return node_features, graph_edge_index


def load_json(path):
    with open(path, "r", encoding="utf-8") as handle:
        return json.load(handle)


def configuration_from_record(record, model_name):
    configuration = record_value(
        record,
        ["configuration", "configuration_json", "selected_configuration", "config", "parameters"],
        {},
    )
    if isinstance(configuration, str):
        configuration = json.loads(configuration)
    fallback = {
        "GCN": {"hidden_dim": 64, "num_layers": 2, "dropout": 0.3, "graph_direction": "directed"},
        "GraphSAGE": {"hidden_dim": 64, "num_layers": 2, "dropout": 0.3, "graph_direction": "directed"},
        "GAT": {"hidden_dim": 32, "num_layers": 2, "dropout": 0.2, "graph_direction": "directed"},
    }[model_name]
    return {**fallback, **configuration}


def checkpoint_state(path):
    payload = torch.load(path, map_location="cpu", weights_only=False)
    if isinstance(payload, dict):
        for name in ["model_state_dict", "state_dict", "model"]:
            if name in payload and isinstance(payload[name], dict):
                return payload[name]
    return payload


def extract_prediction_scores(output):
    if isinstance(output, dict):
        for key in ["scores", "prediction_scores", "probabilities", "edge_scores"]:
            if key in output:
                return output[key]
        for value in output.values():
            if isinstance(value, (np.ndarray, torch.Tensor)) and value.ndim == 1:
                return value
        raise KeyError("The prediction dictionary did not contain a score array")
    if isinstance(output, (tuple, list)):
        for value in output:
            if isinstance(value, (np.ndarray, torch.Tensor)) and value.ndim == 1:
                return value
    return output


print("Step 1 of 7: evaluating converged GNN checkpoints on the final test partition")
converged_rows = []
subset_rows = []

for model_name, model_key in MODEL_KEYS.items():
    for seed in SEED_LIST:
        record_path = ROOT / "gnn_converged" / model_key / "seed_results" / f"seed_{seed}.json"
        checkpoint_path = ROOT / "gnn_converged" / model_key / "checkpoints" / f"{model_key}_seed_{seed}.pt"
        record = load_json(record_path)
        configuration = configuration_from_record(record, model_name)
        threshold = float(record_value(record, ["validation_selected_threshold", "selected_threshold", "threshold"]))
        best_epoch = int(record_value(record, ["best_epoch"], 0))

        graph_run = load_seed_graph(seed)
        node_features, graph_edge_index = graph_components(graph_run)
        node_features = node_features.to(device)
        graph_edge_index = graph_edge_index.to(device)
        node_input_dim = int(node_features.shape[1])
        edge_input_dim = int(test_dense.shape[1])

        model = TransactionEdgeGNN(
            model_name,
            node_input_dim,
            edge_input_dim,
            int(configuration["hidden_dim"]),
            int(configuration["num_layers"]),
            float(configuration["dropout"]),
            configuration.get("graph_direction", "directed"),
        ).to(device)
        model.load_state_dict(checkpoint_state(checkpoint_path))
        model.eval()

        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats(device)
        started = time.perf_counter()
        scores = predict_gnn_scores(
            model,
            node_features,
            graph_edge_index,
            test_sources,
            test_destinations,
            test_dense,
        )
        inference_seconds = time.perf_counter() - started
        scores = extract_prediction_scores(scores)
        if torch.is_tensor(scores):
            scores = scores.detach().cpu().numpy()
        scores = np.asarray(scores, dtype=np.float32)
        metrics, predictions = metric_set(y_test, scores, threshold)
        peak_gpu = torch.cuda.max_memory_allocated(device) / (1024 ** 3) if torch.cuda.is_available() else 0.0

        result = {
            "model": model_name,
            "seed": seed,
            "best_epoch": best_epoch,
            "validation_selected_threshold": threshold,
            **{f"test_{key}": value for key, value in metrics.items()},
            "test_inference_seconds": inference_seconds,
            "test_inference_seconds_per_100000_transactions": inference_seconds * 100000 / len(y_test),
            "peak_gpu_memory_gb_during_test": peak_gpu,
            "configuration_json": json.dumps(configuration, sort_keys=True),
        }
        converged_rows.append(result)

        prediction_frame = pd.DataFrame({
            "transaction_id": np.asarray(test_transaction_ids),
            "timestamp": pd.to_datetime(test_timestamps),
            "actual_label": np.asarray(y_test, dtype=np.int8),
            "score": scores,
            "predicted_label": predictions,
            "is_dense_period": np.asarray(dense_test_mask, dtype=bool),
            "is_timestamp_tail": np.asarray(tail_test_mask, dtype=bool),
        })
        prediction_frame.to_parquet(
            PREDICTIONS / f"{model_key}_test_predictions_seed_{seed}.parquet",
            index=False,
            compression="zstd",
        )

        for subset_name, subset_mask in [
            ("dense_period", np.asarray(dense_test_mask, dtype=bool)),
            ("timestamp_tail", np.asarray(tail_test_mask, dtype=bool)),
        ]:
            subset_metrics, _ = metric_set(np.asarray(y_test)[subset_mask], scores[subset_mask], threshold)
            subset_rows.append({
                "model": model_name,
                "seed": seed,
                "subset": subset_name,
                "transactions": int(subset_mask.sum()),
                "positive_transactions": int(np.asarray(y_test)[subset_mask].sum()),
                **subset_metrics,
            })

        print(
            model_name,
            "seed", seed,
            "F1", f"{metrics['f1']:.6f}",
            "AUPRC", f"{metrics['auprc']:.6f}",
            "ROC-AUC", f"{metrics['roc_auc']:.6f}",
        )
        del model, scores, predictions, prediction_frame
        del graph_run, node_features, graph_edge_index
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

converged_gnn = pd.DataFrame(converged_rows)
converged_gnn.to_csv(TABLES / "converged_gnn_metrics_by_seed.csv", index=False)
pd.DataFrame(subset_rows).to_csv(TABLES / "gnn_dense_and_tail_metrics.csv", index=False)


print("Step 2 of 7: combining all six models and calculating confidence intervals")
conventional_path = ROOT / "conventional_metrics_by_seed.csv"
conventional = pd.read_csv(conventional_path)

rename_map = {}
for column in conventional.columns:
    if column.startswith("test_"):
        continue
    if column in ["accuracy", "precision", "recall", "f1", "auprc", "roc_auc", "true_negatives", "false_positives", "false_negatives", "true_positives"]:
        rename_map[column] = "test_" + column
conventional = conventional.rename(columns=rename_map)
if "test_accuracy" not in conventional.columns:
    total = conventional[["test_true_negatives", "test_false_positives", "test_false_negatives", "test_true_positives"]].sum(axis=1)
    conventional["test_accuracy"] = (conventional["test_true_negatives"] + conventional["test_true_positives"]) / total

metric_columns = ["test_accuracy", "test_precision", "test_recall", "test_f1", "test_auprc", "test_roc_auc"]
combined = pd.concat(
    [conventional[[column for column in conventional.columns if column in set(["model", "seed", "validation_selected_threshold"] + metric_columns + ["test_true_negatives", "test_false_positives", "test_false_negatives", "test_true_positives", "training_seconds", "fixed_inference_seconds", "peak_cpu_memory_gb", "peak_gpu_memory_gb"]) ]], converged_gnn],
    ignore_index=True,
    sort=False,
)
combined.to_csv(TABLES / "all_models_final_metrics_by_seed.csv", index=False)

summary_rows = []
for model_name, group in combined.groupby("model", sort=False):
    for metric in metric_columns:
        values = group[metric].dropna().astype(float).to_numpy()
        mean_value = float(values.mean())
        standard_deviation = float(values.std(ddof=1))
        half_width = float(stats.t.ppf(0.975, len(values) - 1) * standard_deviation / math.sqrt(len(values)))
        summary_rows.append({
            "model": model_name,
            "metric": metric.replace("test_", ""),
            "seeds": len(values),
            "mean": mean_value,
            "standard_deviation": standard_deviation,
            "ci95_lower": mean_value - half_width,
            "ci95_upper": mean_value + half_width,
        })
summary = pd.DataFrame(summary_rows)
summary.to_csv(TABLES / "summary_metrics_with_95_percent_ci.csv", index=False)

wide_summary = summary.pivot(index="model", columns="metric", values="mean").reset_index()
wide_summary.to_csv(TABLES / "summary_metric_means.csv", index=False)
print(wide_summary.to_string(index=False))


print("Step 3 of 7: creating main performance and confusion-matrix figures")
plot_metrics = ["f1", "auprc"]
plot_models = list(wide_summary.sort_values("auprc", ascending=False)["model"])
positions = np.arange(len(plot_models))
width = 0.36
fig, axis = plt.subplots(figsize=(10, 5.5))
for offset, metric in enumerate(plot_metrics):
    metric_frame = summary[summary["metric"] == metric].set_index("model").loc[plot_models]
    means = metric_frame["mean"].to_numpy()
    errors = (metric_frame["ci95_upper"] - metric_frame["ci95_lower"]).to_numpy() / 2
    axis.bar(positions + (offset - 0.5) * width, means, width, yerr=errors, capsize=4, label=metric.upper())
axis.set_xticks(positions)
axis.set_xticklabels(plot_models, rotation=25, ha="right")
axis.set_ylabel("Mean test score across five seeds")
axis.set_title("Transaction-level AML detection performance")
axis.legend()
axis.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURES / "main_performance_comparison.pdf", bbox_inches="tight")
fig.savefig(FIGURES / "main_performance_comparison.png", dpi=350, bbox_inches="tight")
plt.close(fig)

fig, axes = plt.subplots(2, 3, figsize=(12, 7.5))
for axis, model_name in zip(axes.flat, combined["model"].drop_duplicates()):
    group = combined[combined["model"] == model_name].copy()
    target_mean = group["test_f1"].mean()
    representative = group.iloc[(group["test_f1"] - target_mean).abs().argsort().iloc[0]]
    matrix = np.array([
        [representative["test_true_negatives"], representative["test_false_positives"]],
        [representative["test_false_negatives"], representative["test_true_positives"]],
    ], dtype=np.int64)
    image_handle = axis.imshow(np.log10(matrix + 1), cmap="Blues")
    for row_index in range(2):
        for column_index in range(2):
            axis.text(column_index, row_index, f"{matrix[row_index, column_index]:,}", ha="center", va="center", fontsize=9)
    axis.set_title(f"{model_name}, seed {int(representative['seed'])}")
    axis.set_xticks([0, 1], ["Legitimate", "Suspicious"])
    axis.set_yticks([0, 1], ["Legitimate", "Suspicious"])
    axis.set_xlabel("Predicted")
    axis.set_ylabel("Actual")
fig.suptitle("Representative test confusion matrices", y=1.01)
fig.tight_layout()
fig.savefig(FIGURES / "test_confusion_matrices.pdf", bbox_inches="tight")
fig.savefig(FIGURES / "test_confusion_matrices.png", dpi=350, bbox_inches="tight")
plt.close(fig)


print("Step 4 of 7: creating SHAP explanations for the best conventional model")
conventional_validation_column = "validation_auprc" if "validation_auprc" in conventional.columns else None
if conventional_validation_column:
    best_conventional_name = conventional.groupby("model")[conventional_validation_column].mean().idxmax()
else:
    best_conventional_name = summary[summary["metric"] == "auprc"].sort_values("mean", ascending=False).iloc[0]["model"]
    if best_conventional_name in MODEL_KEYS:
        best_conventional_name = "XGBoost"

best_conventional_rows = conventional[conventional["model"] == best_conventional_name]
if conventional_validation_column:
    explanation_seed = int(best_conventional_rows.sort_values(conventional_validation_column, ascending=False).iloc[0]["seed"])
else:
    explanation_seed = int(best_conventional_rows.sort_values("test_auprc", ascending=False).iloc[0]["seed"])

if best_conventional_name != "XGBoost":
    best_conventional_name = "XGBoost"
    explanation_seed = 42

xgb_model_path = ROOT / "models" / "xgboost" / f"xgboost_seed_{explanation_seed}.json"
xgb_model = xgb.XGBClassifier()
xgb_model.load_model(xgb_model_path)
xgb_scores = xgb_model.predict_proba(X_test)[:, 1]
xgb_row = conventional[(conventional["model"] == "XGBoost") & (conventional["seed"] == explanation_seed)].iloc[0]
xgb_threshold = float(xgb_row["validation_selected_threshold"])
xgb_predictions = (xgb_scores >= xgb_threshold).astype(np.int8)

feature_name_candidates = [
    ROOT / "preprocessing" / "processed_feature_names.csv",
    ROOT / "preprocessing" / "feature_names.csv",
    ROOT / "processed_feature_names.csv",
]
feature_names = None
for candidate in feature_name_candidates:
    if candidate.exists():
        name_frame = pd.read_csv(candidate)
        feature_names = name_frame.iloc[:, -1].astype(str).tolist()
        break
if feature_names is None and "preprocessor" in globals():
    try:
        feature_names = list(preprocessor.get_feature_names_out())
    except Exception:
        feature_names = None
if feature_names is None or len(feature_names) != X_test.shape[1]:
    feature_names = [f"feature_{index}" for index in range(X_test.shape[1])]

rng = np.random.default_rng(42)
positive_indices = np.flatnonzero(np.asarray(y_test) == 1)
negative_indices = np.flatnonzero(np.asarray(y_test) == 0)
sample_negative_indices = rng.choice(negative_indices, size=min(5000, len(negative_indices)), replace=False)
shap_indices = np.concatenate([positive_indices, sample_negative_indices])
shap_matrix = X_test[shap_indices].toarray().astype(np.float32)
contributions = xgb_model.get_booster().predict(xgb.DMatrix(shap_matrix, feature_names=None), pred_contribs=True)
shap_values = contributions[:, :-1]
base_values = contributions[:, -1]

mean_abs = np.abs(shap_values).mean(axis=0)
importance = pd.DataFrame({"feature": feature_names, "mean_absolute_shap": mean_abs}).sort_values("mean_absolute_shap", ascending=False)
importance.to_csv(EXPLANATIONS / "xgboost_shap_global_importance.csv", index=False)
top_features = importance.head(20)["feature"].tolist()[::-1]
top_indices = [feature_names.index(name) for name in top_features]
fig, axis = plt.subplots(figsize=(8, 7))
axis.barh(top_features, mean_abs[top_indices])
axis.set_xlabel("Mean absolute SHAP value")
axis.set_title(f"XGBoost global SHAP summary, seed {explanation_seed}")
fig.tight_layout()
fig.savefig(EXPLANATIONS / "xgboost_shap_global_summary.pdf", bbox_inches="tight")
fig.savefig(EXPLANATIONS / "xgboost_shap_global_summary.png", dpi=350, bbox_inches="tight")
plt.close(fig)

true_positive_candidates = np.flatnonzero((np.asarray(y_test) == 1) & (xgb_predictions == 1))
false_negative_candidates = np.flatnonzero((np.asarray(y_test) == 1) & (xgb_predictions == 0))
local_cases = {
    "correctly_detected_suspicious": int(true_positive_candidates[0]),
    "missed_suspicious": int(false_negative_candidates[0]),
}
for case_name, row_index in local_cases.items():
    row_matrix = X_test[row_index].toarray().astype(np.float32)
    row_contribution = xgb_model.get_booster().predict(xgb.DMatrix(row_matrix), pred_contribs=True)[0]
    local_frame = pd.DataFrame({
        "feature": feature_names,
        "feature_value": row_matrix[0],
        "shap_value": row_contribution[:-1],
    })
    local_frame["absolute_shap"] = local_frame["shap_value"].abs()
    local_frame = local_frame.sort_values("absolute_shap", ascending=False)
    local_frame.insert(0, "transaction_id", int(np.asarray(test_transaction_ids)[row_index]))
    local_frame.insert(1, "score", float(xgb_scores[row_index]))
    local_frame.insert(2, "threshold", xgb_threshold)
    local_frame.to_csv(EXPLANATIONS / f"xgboost_shap_{case_name}.csv", index=False)
    plot_frame = local_frame.head(15).sort_values("shap_value")
    fig, axis = plt.subplots(figsize=(8, 6))
    colors = ["#b2182b" if value > 0 else "#2166ac" for value in plot_frame["shap_value"]]
    axis.barh(plot_frame["feature"], plot_frame["shap_value"], color=colors)
    axis.axvline(0, color="black", linewidth=0.8)
    axis.set_xlabel("SHAP contribution to the raw model score")
    axis.set_title(case_name.replace("_", " ").title())
    fig.tight_layout()
    fig.savefig(EXPLANATIONS / f"xgboost_shap_{case_name}.pdf", bbox_inches="tight")
    fig.savefig(EXPLANATIONS / f"xgboost_shap_{case_name}.png", dpi=350, bbox_inches="tight")
    plt.close(fig)


print("Step 5 of 7: preserving and documenting GNN ablations")
ablation_candidates = [
    ROOT / "gnn_ablation_results.csv",
    ROOT / "models" / "gnn" / "gnn_ablation_results.csv",
    ROOT / "gnn_ablation" / "gnn_ablation_results.csv",
]
ablation_source = next((path for path in ablation_candidates if path.exists()), None)
if ablation_source is None and "ablation_results" in globals():
    pd.DataFrame(ablation_results).to_csv(TABLES / "gnn_ablation_results.csv", index=False)
elif ablation_source is not None:
    shutil.copy2(ablation_source, TABLES / "gnn_ablation_results.csv")
else:
    search_results = list(ROOT.rglob("*ablation*.csv"))
    if search_results:
        shutil.copy2(search_results[0], TABLES / "gnn_ablation_results.csv")

ablation_note = (
    "The graph ablation used GraphSAGE and fixed reference seed 42 because of the available session time. "
    "It compared transaction attributes without message passing, directed and symmetrized graphs, and one, two, and three message-passing layers. "
    "The architecture-wide converged comparison subsequently placed GAT narrowly above GraphSAGE by mean validation AUPRC. "
    "This fixed-seed ablation limitation must be reported with the results."
)
(FINAL / "ablation_scope.txt").write_text(clean_text(ablation_note), encoding="utf-8")


print("Step 6 of 7: creating GNNExplainer outputs for the validation-ranked GNN")
gnn_explanation_status = {"status": "not_started"}
try:
    from torch_geometric.explain import Explainer, GNNExplainer
    from torch_geometric.utils import degree, k_hop_subgraph
    import networkx as nx

    validation_frame = pd.read_csv(ROOT / "gnn_converged" / "validation_results.csv")
    best_gnn_name = validation_frame.groupby("model")["validation_auprc"].mean().idxmax()
    best_gnn_key = MODEL_KEYS[best_gnn_name]
    best_seed_row = validation_frame[validation_frame["model"] == best_gnn_name].sort_values("validation_auprc", ascending=False).iloc[0]
    best_gnn_seed = int(best_seed_row["seed"])
    best_record = load_json(ROOT / "gnn_converged" / best_gnn_key / "seed_results" / f"seed_{best_gnn_seed}.json")
    best_configuration = configuration_from_record(best_record, best_gnn_name)
    best_threshold = float(record_value(best_record, ["validation_selected_threshold", "selected_threshold", "threshold"]))
    best_predictions_path = PREDICTIONS / f"{best_gnn_key}_test_predictions_seed_{best_gnn_seed}.parquet"
    best_predictions = pd.read_parquet(best_predictions_path)

    graph_run = load_seed_graph(best_gnn_seed)
    node_features, graph_edge_index = graph_components(graph_run)
    node_features = node_features.to(device)
    graph_edge_index = graph_edge_index.to(device)
    base_model = TransactionEdgeGNN(
        best_gnn_name,
        int(node_features.shape[1]),
        int(test_dense.shape[1]),
        int(best_configuration["hidden_dim"]),
        int(best_configuration["num_layers"]),
        float(best_configuration["dropout"]),
        best_configuration.get("graph_direction", "directed"),
    ).to(device)
    best_checkpoint = ROOT / "gnn_converged" / best_gnn_key / "checkpoints" / f"{best_gnn_key}_seed_{best_gnn_seed}.pt"
    base_model.load_state_dict(checkpoint_state(best_checkpoint))
    base_model.eval()

    class EdgeExplanationWrapper(torch.nn.Module):
        def __init__(self, wrapped_model):
            super().__init__()
            self.wrapped_model = wrapped_model

        def forward(self, x, edge_index, edge_label_index, target_edge_features):
            return self.wrapped_model(
                x,
                edge_index,
                edge_label_index[0],
                edge_label_index[1],
                target_edge_features,
            )

    full_edge_index_cpu = graph_edge_index.detach().cpu() if torch.is_tensor(graph_edge_index) else torch.as_tensor(graph_edge_index)
    node_degree = degree(full_edge_index_cpu.reshape(-1), num_nodes=int(node_features.shape[0]))
    actual = best_predictions["actual_label"].to_numpy()
    predicted = best_predictions["predicted_label"].to_numpy()
    case_candidates = {
        "correctly_detected_suspicious": np.flatnonzero((actual == 1) & (predicted == 1)),
        "missed_suspicious": np.flatnonzero((actual == 1) & (predicted == 0)),
    }

    for case_name, candidates in case_candidates.items():
        if len(candidates) == 0:
            raise RuntimeError("No transaction was available for " + case_name)
        candidate_subset = candidates[: min(500, len(candidates))]
        endpoint_cost = (
            node_degree[torch.as_tensor(test_sources[candidate_subset], dtype=torch.long)]
            + node_degree[torch.as_tensor(test_destinations[candidate_subset], dtype=torch.long)]
        )
        row_index = int(candidate_subset[int(torch.argmin(endpoint_cost))])
        source_node = int(test_sources[row_index])
        destination_node = int(test_destinations[row_index])

        subset, local_edge_index, mapping, original_edge_mask = k_hop_subgraph(
            torch.tensor([source_node, destination_node], dtype=torch.long),
            num_hops=int(best_configuration["num_layers"]),
            edge_index=full_edge_index_cpu,
            relabel_nodes=True,
            directed=False,
        )
        local_x = node_features[subset].to(device)
        local_edge_index = local_edge_index.to(device)
        target_edge_index = mapping.reshape(2, 1).to(device)
        target_feature = torch.as_tensor(test_dense[row_index:row_index + 1], dtype=torch.float32, device=device)

        wrapper = EdgeExplanationWrapper(base_model).to(device)
        explainer = Explainer(
            model=wrapper,
            algorithm=GNNExplainer(epochs=100, lr=0.01),
            explanation_type="model",
            node_mask_type="attributes",
            edge_mask_type="object",
            model_config={
                "mode": "binary_classification",
                "task_level": "edge",
                "return_type": "raw",
            },
        )
        explanation = explainer(
            x=local_x,
            edge_index=local_edge_index,
            edge_label_index=target_edge_index,
            target_edge_features=target_feature,
            index=0,
        )
        edge_mask_values = explanation.edge_mask.detach().cpu().numpy()
        original_edges = full_edge_index_cpu[:, original_edge_mask].numpy()
        edge_frame = pd.DataFrame({
            "source_node": original_edges[0],
            "destination_node": original_edges[1],
            "importance": edge_mask_values,
        }).sort_values("importance", ascending=False)
        edge_frame.insert(0, "explained_transaction_id", int(test_transaction_ids[row_index]))
        edge_frame.insert(1, "explained_score", float(best_predictions.iloc[row_index]["score"]))
        edge_frame.insert(2, "validation_threshold", best_threshold)
        edge_frame.to_csv(EXPLANATIONS / f"{best_gnn_key}_gnnexplainer_{case_name}_edges.csv", index=False)

        if explanation.node_mask is not None:
            node_mask_array = explanation.node_mask.detach().cpu().numpy()
            pd.DataFrame(node_mask_array).to_csv(
                EXPLANATIONS / f"{best_gnn_key}_gnnexplainer_{case_name}_node_feature_mask.csv",
                index=False,
            )

        top_edges = edge_frame.head(35)
        graph_plot = nx.DiGraph()
        for _, edge_row in top_edges.iterrows():
            graph_plot.add_edge(int(edge_row["source_node"]), int(edge_row["destination_node"]), weight=float(edge_row["importance"]))
        graph_plot.add_node(source_node)
        graph_plot.add_node(destination_node)
        layout = nx.spring_layout(graph_plot, seed=42)
        widths = [0.5 + 4.0 * graph_plot[u][v]["weight"] for u, v in graph_plot.edges()]
        node_colors = [
            "#d73027" if node == source_node else "#4575b4" if node == destination_node else "#cccccc"
            for node in graph_plot.nodes()
        ]
        fig, axis = plt.subplots(figsize=(9, 7))
        nx.draw_networkx(
            graph_plot,
            pos=layout,
            ax=axis,
            with_labels=False,
            node_size=90,
            node_color=node_colors,
            width=widths,
            arrows=True,
            alpha=0.85,
        )
        axis.set_title(f"{best_gnn_name} GNNExplainer: {case_name.replace('_', ' ')}")
        axis.axis("off")
        fig.tight_layout()
        fig.savefig(EXPLANATIONS / f"{best_gnn_key}_gnnexplainer_{case_name}.pdf", bbox_inches="tight")
        fig.savefig(EXPLANATIONS / f"{best_gnn_key}_gnnexplainer_{case_name}.png", dpi=350, bbox_inches="tight")
        plt.close(fig)

    gnn_explanation_status = {
        "status": "complete",
        "model": best_gnn_name,
        "seed": best_gnn_seed,
        "past_only_graph": True,
        "future_validation_or_test_edges_present": False,
    }
    del graph_run, node_features, graph_edge_index, base_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception as explanation_error:
    gnn_explanation_status = {"status": "error", "message": clean_text(repr(explanation_error))}
    print("GNNExplainer needs a small follow-up correction:", repr(explanation_error))

with open(EXPLANATIONS / "gnnexplainer_status.json", "w", encoding="utf-8") as handle:
    json.dump(gnn_explanation_status, handle, indent=2)


print("Step 7 of 7: recording environment, reconstructing the notebook, checking text, and building ZIP")
environment = {
    "cpu": platform.processor(),
    "ram_gb": round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / (1024 ** 3), 3),
    "operating_system": platform.platform(),
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "scipy": scipy.__version__,
    "scikit_learn": sklearn.__version__,
    "xgboost": xgb.__version__,
    "pytorch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "gpu_memory_gb": round(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 3) if torch.cuda.is_available() else 0,
}
try:
    import torch_geometric
    environment["torch_geometric"] = torch_geometric.__version__
except Exception:
    pass
with open(FINAL / "software_and_hardware_environment.json", "w", encoding="utf-8") as handle:
    json.dump(environment, handle, indent=2)

method_note = """FINAL EXPERIMENT PROTOCOL

Dataset: complete IBM AML HI-Small transaction file after removal of nine exact duplicate rows.
Prediction unit: transaction-level suspicious label for all six models.
Split: chronological, approximately 60 percent training, 20 percent validation, and 20 percent test. Identical timestamps remain in one partition.
Preprocessing: fitted using training data only. Validation and test data retain their natural class distributions.
Training sample: every training positive and a seed-specific 50 to 1 negative sample. The same seed-specific transaction sample is used across model families.
Threshold: selected using validation data only.
Graph: accounts are nodes and transactions are directed edges. Account identifiers are used only for connectivity. Validation and test edges are excluded from message passing.
Graph prediction: account embeddings are combined with transaction attributes by an edge decoder.
Historical features: computed from transactions with strictly earlier timestamps. Transactions sharing the target timestamp do not contribute to one another.
Primary imbalance metric: suspicious-class AUPRC. Suspicious-class precision, recall, F1, ROC-AUC, accuracy, and raw confusion counts are also reported.
Repeated runs: seeds 42, 52, 62, 72, and 82. Summary tables report the mean, sample standard deviation, and two-sided 95 percent t confidence interval.
Timestamp anomaly: the sparse high-positive-rate tail from September 11 onward is retained in the full test result and reported separately as a sensitivity analysis.
Earlier eight-epoch graph runs were diagnostic pilots. They were excluded from final performance tables. Converged graph model selection used validation AUPRC only.
"""
(FINAL / "experiment_protocol.txt").write_text(clean_text(method_note), encoding="utf-8")

readme = """AML EXPERIMENT OUTPUTS

This directory contains the final transaction-level evaluation for Random Forest, XGBoost, Linear SVM, GCN, GraphSAGE, and GAT on IBM AML HI-Small.

Important tables are in final_delivery/tables. Main figures are in final_delivery/figures. SHAP and GNNExplainer outputs are in final_delivery/explainability. Compressed test predictions are in final_delivery/predictions.

The full audit, split manifests, preprocessing configuration, search records, checkpoints, and intermediate records are stored in the surrounding aml_outputs directories and are included in the ZIP where practical. Large reusable matrices and graph caches are excluded because they can be regenerated from the included notebook and source dataset.

Do not substitute results from the earlier manuscript. Only the files labelled final or converged should be used for revision.
"""
(FINAL / "README.txt").write_text(clean_text(readme), encoding="utf-8")

try:
    import nbformat
    shell = get_ipython()
    raw_cells = [cell for cell in shell.history_manager.input_hist_raw[1:] if cell.strip()]
    notebook = nbformat.v4.new_notebook()
    notebook.cells = [nbformat.v4.new_code_cell(clean_text(cell)) for cell in raw_cells]
    notebook.metadata["kernelspec"] = {"display_name": "Python 3", "language": "python", "name": "python3"}
    nbformat.write(notebook, FINAL / "AML_complete_experiment_notebook.ipynb")
except Exception as notebook_error:
    (FINAL / "notebook_export_error.txt").write_text(clean_text(repr(notebook_error)), encoding="utf-8")

scan_roots = [FINAL]
text_suffixes = {".txt", ".csv", ".json", ".md", ".tex", ".py", ".ipynb"}
scan_issues = []
for scan_root in scan_roots:
    for path in scan_root.rglob("*"):
        if path.is_file() and path.suffix.lower() in text_suffixes:
            try:
                content = path.read_text(encoding="utf-8")
            except Exception:
                continue
            if chr(8212) in content:
                scan_issues.append({"file": str(path), "issue": "em_dash_character"})
            if "-" * 3 in content:
                scan_issues.append({"file": str(path), "issue": "triple_hyphen_sequence"})
pd.DataFrame(scan_issues, columns=["file", "issue"]).to_csv(FINAL / "text_compliance_scan.csv", index=False)
if scan_issues:
    raise RuntimeError("Text compliance scan found prohibited punctuation. See text_compliance_scan.csv")

include_directories = [
    FINAL,
    ROOT / "audit",
    ROOT / "preprocessing",
    ROOT / "models",
    ROOT / "gnn_converged",
]
include_files = [
    ROOT / "dataset_audit.json",
    ROOT / "split_summary.csv",
    ROOT / "feature_summary.csv",
    ROOT / "training_sampling_summary.csv",
    ROOT / "conventional_metrics_by_seed.csv",
    ROOT / "gnn_metrics_by_seed.csv",
]
zip_path = ROOT / "AML_IEEE_revision_complete_outputs.zip"
if zip_path.exists():
    zip_path.unlink()
excluded_parts = {"matrices", "graph_cache", "training_samples", "__pycache__"}
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    added = set()
    for directory in include_directories:
        if not directory.exists():
            continue
        for path in directory.rglob("*"):
            if not path.is_file() or any(part in excluded_parts for part in path.parts):
                continue
            relative = path.relative_to(ROOT)
            if str(relative) not in added:
                archive.write(path, arcname=str(relative))
                added.add(str(relative))
    for path in include_files:
        if path.exists():
            relative = path.relative_to(ROOT)
            if str(relative) not in added:
                archive.write(path, arcname=str(relative))
                added.add(str(relative))

print("FINAL PACKAGE COMPLETE")
print("Best conventional explanation model:", best_conventional_name, "seed", explanation_seed)
print("GNNExplainer status:", gnn_explanation_status)
print("Final ZIP:", zip_path)
print("ZIP size in GB:", round(zip_path.stat().st_size / (1024 ** 3), 3))
print("Send me the final printed lines and the summary metric table.")

## Completed GNNExplainer outputs

Creates the corrected past-only GNN explanations and rebuilds the final archive.


In [ ]:
from pathlib import Path
import gc
import json
import shutil
import zipfile

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import torch
from torch_geometric.explain import Explainer, GNNExplainer
from torch_geometric.utils import degree, k_hop_subgraph

ROOT = Path("/kaggle/working/aml_outputs")
FINAL = ROOT / "final_delivery"
EXPLANATIONS = FINAL / "explainability"
PREDICTIONS = FINAL / "predictions"
EXPLANATIONS.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


def value_from_record(record, names, default=None):
    for name in names:
        if name in record:
            return record[name]
    return default


def configuration_from_file(record, model_name):
    configuration = value_from_record(
        record,
        ["configuration", "configuration_json", "selected_configuration", "config", "parameters"],
        {},
    )
    if isinstance(configuration, str):
        configuration = json.loads(configuration)
    fallback = {
        "GCN": {"hidden_dim": 64, "num_layers": 2, "dropout": 0.3, "graph_direction": "directed"},
        "GraphSAGE": {"hidden_dim": 64, "num_layers": 2, "dropout": 0.3, "graph_direction": "directed"},
        "GAT": {"hidden_dim": 32, "num_layers": 2, "dropout": 0.2, "graph_direction": "directed"},
    }[model_name]
    return {**fallback, **configuration}


def state_from_checkpoint(path):
    payload = torch.load(path, map_location="cpu", weights_only=False)
    if isinstance(payload, dict):
        for name in ["model_state_dict", "state_dict", "model"]:
            if name in payload and isinstance(payload[name], dict):
                return payload[name]
    return payload


def component_from_graph(graph_run, names, predicate):
    if isinstance(graph_run, dict):
        for name in names:
            if name in graph_run:
                return graph_run[name]
        values = graph_run.values()
    else:
        for name in names:
            if hasattr(graph_run, name):
                return getattr(graph_run, name)
        values = graph_run if isinstance(graph_run, (tuple, list)) else []
    for value in values:
        try:
            if predicate(value):
                return value
        except Exception:
            pass
    raise KeyError("Required graph component was not found")


def load_graph_parts(seed):
    graph_run = load_seed_graph(seed)
    node_features = component_from_graph(
        graph_run,
        ["node_features", "node_x", "x"],
        lambda item: hasattr(item, "ndim") and item.ndim == 2 and item.shape[1] == 108,
    )
    edge_index = component_from_graph(
        graph_run,
        ["graph_edge_index", "message_edge_index", "edge_index"],
        lambda item: hasattr(item, "ndim") and item.ndim == 2 and item.shape[0] == 2 and item.shape[1] > 1000000,
    )
    return graph_run, node_features, edge_index


print("Selecting the GNN using mean validation AUPRC only...")
validation_frame = pd.read_csv(ROOT / "gnn_converged" / "validation_results.csv")
mean_validation = validation_frame.groupby("model", as_index=False)["validation_auprc"].mean()
best_gnn_name = str(mean_validation.sort_values("validation_auprc", ascending=False).iloc[0]["model"])
model_key = {"GCN": "gcn", "GraphSAGE": "graphsage", "GAT": "gat"}[best_gnn_name]
best_seed_row = validation_frame[validation_frame["model"] == best_gnn_name].sort_values(
    "validation_auprc", ascending=False
).iloc[0]
best_seed = int(best_seed_row["seed"])

record_path = ROOT / "gnn_converged" / model_key / "seed_results" / f"seed_{best_seed}.json"
with open(record_path, "r", encoding="utf-8") as handle:
    record = json.load(handle)
configuration = configuration_from_file(record, best_gnn_name)
threshold = float(value_from_record(record, ["validation_selected_threshold", "selected_threshold", "threshold"]))

prediction_path = PREDICTIONS / f"{model_key}_test_predictions_seed_{best_seed}.parquet"
test_prediction_frame = pd.read_parquet(prediction_path)

for stale_name in [
    "graph_run",
    "base_model",
    "wrapper",
    "explainer",
    "explanation",
    "node_features",
    "graph_edge_index",
    "local_x",
    "local_edge_index",
    "full_edge_index_cpu",
]:
    if stale_name in globals():
        try:
            del globals()[stale_name]
        except Exception:
            pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

graph_run, node_features_cpu, edge_index_cpu = load_graph_parts(best_seed)
node_features_cpu = node_features_cpu.detach().cpu()
edge_index_cpu = edge_index_cpu.detach().cpu()

base_model = TransactionEdgeGNN(
    best_gnn_name,
    int(node_features_cpu.shape[1]),
    int(test_dense.shape[1]),
    int(configuration["hidden_dim"]),
    int(configuration["num_layers"]),
    float(configuration["dropout"]),
    configuration.get("graph_direction", "directed"),
).to(device)
checkpoint_path = ROOT / "gnn_converged" / model_key / "checkpoints" / f"{model_key}_seed_{best_seed}.pt"
base_model.load_state_dict(state_from_checkpoint(checkpoint_path))
base_model.eval()


class EdgeExplanationWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x, edge_index, target_edge_index, target_edge_features):
        node_embeddings = self.model.encode(x, edge_index)
        return self.model.decode(
            node_embeddings,
            target_edge_index,
            target_edge_features,
        )


node_degree = degree(edge_index_cpu.reshape(-1), num_nodes=int(node_features_cpu.shape[0]))
actual_labels = test_prediction_frame["actual_label"].to_numpy()
predicted_labels = test_prediction_frame["predicted_label"].to_numpy()
case_candidates = {
    "correctly_detected_suspicious": np.flatnonzero((actual_labels == 1) & (predicted_labels == 1)),
    "missed_suspicious": np.flatnonzero((actual_labels == 1) & (predicted_labels == 0)),
}

case_summary = []
for case_name, candidates in case_candidates.items():
    if len(candidates) == 0:
        raise RuntimeError("No eligible transaction exists for " + case_name)

    candidate_subset = candidates[: min(500, len(candidates))]
    candidate_sources = torch.as_tensor(np.asarray(test_sources)[candidate_subset], dtype=torch.long)
    candidate_destinations = torch.as_tensor(np.asarray(test_destinations)[candidate_subset], dtype=torch.long)
    endpoint_cost = node_degree[candidate_sources] + node_degree[candidate_destinations]
    row_index = int(candidate_subset[int(torch.argmin(endpoint_cost))])
    source_node = int(np.asarray(test_sources)[row_index])
    destination_node = int(np.asarray(test_destinations)[row_index])

    subset, local_edge_index, mapping, original_edge_mask = k_hop_subgraph(
        torch.tensor([source_node, destination_node], dtype=torch.long),
        num_hops=int(configuration["num_layers"]),
        edge_index=edge_index_cpu,
        relabel_nodes=True,
        directed=False,
    )
    local_node_features = node_features_cpu[subset].to(device)
    local_edge_index = local_edge_index.to(device)
    local_target_edge_index = mapping.reshape(2, 1).to(device)
    local_target_features = torch.as_tensor(
        np.asarray(test_dense[row_index:row_index + 1]),
        dtype=torch.float32,
        device=device,
    )

    wrapper = EdgeExplanationWrapper(base_model).to(device)
    explainer = Explainer(
        model=wrapper,
        algorithm=GNNExplainer(epochs=150, lr=0.01),
        explanation_type="model",
        node_mask_type="attributes",
        edge_mask_type="object",
        model_config={
            "mode": "binary_classification",
            "task_level": "edge",
            "return_type": "raw",
        },
    )
    explanation = explainer(
        x=local_node_features,
        edge_index=local_edge_index,
        target_edge_index=local_target_edge_index,
        target_edge_features=local_target_features,
        index=0,
    )

    edge_importance = explanation.edge_mask.detach().cpu().numpy()
    original_edges = edge_index_cpu[:, original_edge_mask].numpy()
    edge_frame = pd.DataFrame({
        "source_node": original_edges[0],
        "destination_node": original_edges[1],
        "importance": edge_importance,
    }).sort_values("importance", ascending=False)
    transaction_id = int(np.asarray(test_transaction_ids)[row_index])
    score = float(test_prediction_frame.iloc[row_index]["score"])
    edge_frame.insert(0, "explained_transaction_id", transaction_id)
    edge_frame.insert(1, "explained_score", score)
    edge_frame.insert(2, "validation_threshold", threshold)
    edge_frame.to_csv(EXPLANATIONS / f"{model_key}_gnnexplainer_{case_name}_edges.csv", index=False)

    if explanation.node_mask is not None:
        node_mask = explanation.node_mask.detach().cpu().numpy()
        pd.DataFrame(node_mask).to_csv(
            EXPLANATIONS / f"{model_key}_gnnexplainer_{case_name}_node_feature_mask.csv",
            index=False,
        )

    top_edges = edge_frame.head(35)
    display_graph = nx.DiGraph()
    for _, edge_row in top_edges.iterrows():
        display_graph.add_edge(
            int(edge_row["source_node"]),
            int(edge_row["destination_node"]),
            weight=float(edge_row["importance"]),
        )
    display_graph.add_node(source_node)
    display_graph.add_node(destination_node)
    layout = nx.spring_layout(display_graph, seed=42)
    widths = [0.5 + 4.0 * display_graph[u][v]["weight"] for u, v in display_graph.edges()]
    colors = [
        "#d73027" if node == source_node else "#4575b4" if node == destination_node else "#cccccc"
        for node in display_graph.nodes()
    ]
    fig, axis = plt.subplots(figsize=(9, 7))
    nx.draw_networkx(
        display_graph,
        pos=layout,
        ax=axis,
        with_labels=False,
        node_size=90,
        node_color=colors,
        width=widths,
        arrows=True,
        alpha=0.85,
    )
    axis.set_title(f"{best_gnn_name} GNNExplainer: {case_name.replace('_', ' ')}")
    axis.axis("off")
    fig.tight_layout()
    fig.savefig(EXPLANATIONS / f"{model_key}_gnnexplainer_{case_name}.pdf", bbox_inches="tight")
    fig.savefig(EXPLANATIONS / f"{model_key}_gnnexplainer_{case_name}.png", dpi=350, bbox_inches="tight")
    plt.close(fig)

    case_summary.append({
        "case": case_name,
        "model": best_gnn_name,
        "seed": best_seed,
        "transaction_id": transaction_id,
        "actual_label": 1,
        "predicted_label": int(predicted_labels[row_index]),
        "score": score,
        "validation_threshold": threshold,
        "local_nodes": int(len(subset)),
        "local_message_edges": int(local_edge_index.shape[1]),
        "past_only_graph": True,
        "future_validation_or_test_edges_present": False,
    })
    print(
        case_name,
        "complete with",
        len(subset),
        "local nodes and",
        local_edge_index.shape[1],
        "message edges",
    )

pd.DataFrame(case_summary).to_csv(EXPLANATIONS / "gnnexplainer_case_summary.csv", index=False)
status = {
    "status": "complete",
    "model": best_gnn_name,
    "seed": best_seed,
    "selection_rule": "highest mean validation AUPRC, then highest seed-level validation AUPRC",
    "past_only_graph": True,
    "future_validation_or_test_edges_present": False,
}
with open(EXPLANATIONS / "gnnexplainer_status.json", "w", encoding="utf-8") as handle:
    json.dump(status, handle, indent=2)

del wrapper, explainer, explanation, base_model, graph_run
del node_features_cpu, edge_index_cpu, local_node_features, local_edge_index
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Rebuilding the ZIP with the completed explanations...")
zip_path = ROOT / "AML_IEEE_revision_complete_outputs.zip"
if zip_path.exists():
    zip_path.unlink()

include_directories = [
    FINAL,
    ROOT / "audit",
    ROOT / "preprocessing",
    ROOT / "models",
    ROOT / "gnn_converged",
]
include_files = [
    ROOT / "dataset_audit.json",
    ROOT / "split_summary.csv",
    ROOT / "feature_summary.csv",
    ROOT / "training_sampling_summary.csv",
    ROOT / "conventional_metrics_by_seed.csv",
    ROOT / "gnn_metrics_by_seed.csv",
]
excluded_parts = {"matrices", "graph_cache", "training_samples", "__pycache__"}
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    added = set()
    for directory in include_directories:
        if not directory.exists():
            continue
        for path in directory.rglob("*"):
            if not path.is_file() or any(part in excluded_parts for part in path.parts):
                continue
            relative = path.relative_to(ROOT)
            if str(relative) not in added:
                archive.write(path, arcname=str(relative))
                added.add(str(relative))
    for path in include_files:
        if path.exists():
            relative = path.relative_to(ROOT)
            if str(relative) not in added:
                archive.write(path, arcname=str(relative))
                added.add(str(relative))

print("GNNEXPLAINER COMPLETE")
print("Model:", best_gnn_name)
print("Seed:", best_seed)
print("Status:", status)
print("Updated ZIP:", zip_path)
print("ZIP size in GB:", round(zip_path.stat().st_size / (1024 ** 3), 3))